# Durable Alignment in LLMs via Orthogonal Memory — v2

**Companion notebook to the paper _Durable Alignment via Orthogonal Memory in Frozen Language Models_.**

This is the **claim-driven rebuild** (v2) of the original organic-research notebook. The eight
claims are organised into a four-layer dependency stack (Existence → Properties → Operations →
Composition); each section produces one piece of empirical evidence and writes its artifacts
to `mmlu_ibf_out/` under the `cN_*` naming scheme.

## Central thesis

> _IBF is a local durable alignment substrate for frozen LLMs whose properties (decoupling,
> generality, distinction) support a clean operational lifecycle (install / revise / remove
> with preserved locality) which in turn supports complementary deductive and inductive
> composition paths — without modifying base-model weights._

## Four-layer stack

```
Layer 4 — COMPOSITION
   C7  Compiled closure (deductive)
   C8  Discovery-driven extension (inductive)
        — includes Crucible-adjudicated conflict resolution (D8 evidence)

Layer 3 — OPERATIONS
   C5  Truth-maintenance lifecycle
   C6  Locality preservation

Layer 2 — PROPERTIES
   C2  Substrate decoupling under base evolution
   C3  Cross-model mechanism generality
   C4  Distinction from kNN/RAG

Layer 1 — EXISTENCE
   C1  Local durable alignment without weight editing
```

Each upper layer is _earned_ by the layers below it. Falsifying a lower claim cascades upward.

## Foundational anchoring

Each claim in this notebook is the LLM-substrate realisation of a concept from the
foundational paper *Information as Alignment v1* (Sections 2–4, 5.4, 8.1). The mapping:

| Companion claim | Foundational anchor |
|---|---|
| C1 — Existence | Foundational Claim 1 (Memory) |
| C2 — Decoupling | Postulate 1 constraint (iii) |
| C3 — Generality | Foundational Claim 5 (Discrete Convergence) |
| C4 — Distinction | Foundational § 8.2 |
| C5 — Lifecycle | Foundational Claims 1 + 4 (Memory + Self-Correction) |
| C6 — Locality | Postulate 1 localisation kernel `K(y, x_S)` |
| C7 — Deductive composition | Foundational § 7.2 + § 5.4 LLM-extension flag |
| C8 — Inductive composition | Foundational Claims 2 + 3 (Agency + Intelligence) + § 8.1 regime-dependence |

## Reading order

Cells are intended to be run top to bottom. Setup (S1–S3) builds the engine, geometry, and
dataset; the eight claim sections each consume those globals and write their artifacts; the
two synthesis sections (S4–S5) aggregate the JSON outputs into a single paper-deliverable.

The previous organic notebook `(IBF)Companion-LLM-Durable-Alignment.ipynb` is preserved as
the paper-run historical record. This v2 is the canonical artifact going forward.


## Run configuration

`RUN_MODE` controls every long-running cell. Three values are supported:

- `"smoke"` — minimum epochs, small datasets, smoke caps. ~30 min end-to-end on an A100.
- `"paper"` — full paper-grade run with standard hard caps. ~35h+ end-to-end.
- `"verify-convergence"` — paper-grade caps **with** the strong-convergence early-stop
  protocol enabled (see Part 6 of the handover). This is the C1 validation-gate mode.
  Estimated ~2.5× compute reduction if the gate passes.

Headline numbers below are the C1 validation-gate references — used in every
long-running cell's `EXPECTED / GOT / WITHIN_TOLERANCE` log line.


In [ ]:
# ════════════════════════════════════════════════════════════════
# Top-level run configuration
# ════════════════════════════════════════════════════════════════
# RUN_MODE: one of {"smoke", "paper", "verify-convergence"}.
# SEED: global seed; each long-running cell uses SEED + offset.
# ════════════════════════════════════════════════════════════════

import os, datetime

RUN_MODE = os.environ.get("IBF_RUN_MODE", "paper")
assert RUN_MODE in ("smoke", "paper", "verify-convergence"), f"Bad RUN_MODE: {RUN_MODE}"

SEED = 42

# Per-claim seed offsets — keeps each long-running cell deterministic and independent.
SEED_OFFSETS = {
    "C1": 100, "C2": 200, "C3": 300, "C4": 400,
    "C5": 500, "C6": 600, "C7": 700, "C8": 800,
}

# C1 validation-gate reference (handover Part 6).
# avg lin from the paper-run canonical_training_results.json.
HEADLINE_AVG_LIN = 0.954
HEADLINE_AVG_LIN_TOL = 0.01  # gate threshold: ±0.01 absolute

# Convergence-stop protocol — gated by RUN_MODE.
EARLY_STOP_STRONG_CONVERGE = (RUN_MODE == "verify-convergence")

# Pin run identifier for log/artifact naming.
RUN_ID = os.environ.get(
    "IBF_RUN_ID",
    datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ"),
)

print(f"RUN_MODE                       = {RUN_MODE!r}")
print(f"SEED                           = {SEED}")
print(f"EARLY_STOP_STRONG_CONVERGE     = {EARLY_STOP_STRONG_CONVERGE}")
print(f"HEADLINE_AVG_LIN               = {HEADLINE_AVG_LIN} ± {HEADLINE_AVG_LIN_TOL}")
print(f"RUN_ID                         = {RUN_ID}")


# Part I — Setup (S1 / S2 / S3)

Construct the deterministic environment: the universal engine (with the Reading C agency
patch), the representation geometry, and the Future Industries dataset. After Part I, every
subsequent claim section consumes the same `ibf` object, `precomputed` dict, and
`phase_data` structure.

---

## § S1 — Universal IBF engine (with Reading C patch)

**Objective.** Define the universal IBF engine as a particle approximation of the
foundational paper's Modification Postulate (Section 4 of the foundation paper). The engine
is **domain-agnostic**: it operates on a d-dimensional latent space supplied by a frozen
encoder and a baseline evaluator. Every modification dynamic below corresponds to a specific
equation or constraint in Postulate 1.

**Mapping to Postulate 1.**

| Engine method | Postulate 1 component |
|---|---|
| `MemoryCenter` | particle `c_i = (z_i, a_i, σ_i, μ_eff,i, ctx_i, …)` (foundation § 4.1) |
| `kernel_batch` | localisation kernel `K(y, x_S) = exp(-‖y - x_S‖² / 2σ²)` (Postulate 1, constraint i) |
| `_read_gate` | reflexive read-gating `γ_i` (foundation § 4.2, Eq. 9) |
| `delta_R` | kernel readout `δR̂(z) = Σ γ_i v_i K(z, z_i)` (Eq. 7) |
| `delta_k` | **intensive** responsiveness readout `δk(z) = Σ I·γ·w·K / Σ I·γ·K` (Eq. 8) |
| `_update_value` | write path Pass 2, same-context spatial learning (Eq. 11) |
| `_update_agency` | parallel equation for responsiveness modification (Postulate 1) |
| `_crystallize` | stability transition `μ_eff → μ_cryst` under exposure + convergence |
| `_crucible` | contradiction-triggered dissolution `v_i · D̄_raw < θ_rev` (Eq. 13) |
| `_merge_pop` | merge + capacity control under dynamic spatial threshold |

### Reading C patch (mandatory)

The original engine gates the agency channel's `w` update and `δk` readout on
crystallisation. This is a conservative discretisation that under-implements Postulate 1's
**parallel equation** framing (foundation § 3.3, where the paper explicitly notes _"A
parallel equation governs the responsiveness modification δk_S (the agency channel)"_).

The Reading C refinement gates the agency channel on **history sufficiency**
(`len(c.D_history) ≥ n_agency_min`) rather than on crystallisation. This brings the agency
channel into structural parity with the value channel: transient `w` updates with a slower
learning rate (`eta_k_cryst`), and transient `w` contributes to `δk` readout. The change is
behaviourally inert on the large-data regimes from the foundational paper (chess / RRW /
CIFAR — see foundation § 8.1) where crystallisation fires within thousands of updates, but
it is the only way to make the agency channel testable on the small-data LLM substrate.

**Empirical justification (D6 result).**

Under the patched engine, the β k_eff at A→C queries traces a U-shape (4.66 → 2.58 → 3.23)
tracking the local D-variance dynamics, while the α (status-quo) k_eff stays flat at the
baseline `k_0 = 5.000`. This U-shape is the empirical signature of Postulate 1's parallel
equation: responsiveness rises in low-variance regions (where corrections converge) and
drops in high-variance regions (where alignment is uncertain). See
`AGENCY_DISCRETIZATION_NOTE.md` and `mmlu_ibf_out/fi_agency_channel_d6_alpha_vs_beta.json`
for the supervisor's pre-merge validation transcript.

**Pre-merge validation (handover Part 7).** C1 with the patched engine must match unpatched
within ±0.01 on avg lin; C3 (Qwen) must match within ±0.02 on all 6 cross-model metrics.
Both gates are checked downstream in this notebook.


In [ ]:
# ════════════════════════════════════════════════════════════════
# § S1 — Universal IBF engine (with Reading C patch)
# Layer: setup
# Presupposes: nothing
# Artifacts: none (defines IBFParams, MemoryCenter, IBFEngine)
# Convergence-stop: n/a
# ════════════════════════════════════════════════════════════════
# This is the domain-agnostic engine that instantiates Postulate 1 of the
# foundation paper. Every modification dynamic below maps to a specific
# equation in foundation § 4.
#
# Reading C patch (mandatory): the agency channel is gated by history
# sufficiency rather than crystallisation. See the markdown above for the
# full justification.
# ════════════════════════════════════════════════════════════════

import torch, torch.nn.functional as F, numpy as np
import time, os, json, random
from dataclasses import dataclass, field
from typing import List, Dict
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
OUT_DIR = "mmlu_ibf_out"; os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Versioned engine identifier — written into every C-cell artifact's metadata.
ENGINE_VERSION = "2.0-history_gate"


@dataclass
class IBFParams:
    sigma: float = 0.206
    merge_threshold: float = 0.366
    sigma_agency: float = None
    eta: float = 0.1
    eta_cryst: float = 0.005
    eta_k: float = 0.05
    # Reading C addition: parallel learning rate for crystallised agency.
    eta_k_cryst: float = 0.005
    mu_base: float = 0.06
    mu_crystallized: float = 0.001
    v_max: float = 0.50
    w_max: float = 5.0
    k_0: float = 5.0
    k_min: float = 1.0
    crystallization_threshold: int = 15
    convergence_threshold: float = 0.025
    # Reading C addition: history sufficiency gate for agency channel.
    n_agency_min: int = 20
    n_cross_min: int = 8
    reversal_threshold: float = -0.0125
    w_dvar_threshold: float = 0.005
    activation_threshold: float = 0.01
    creation_threshold: float = 0.01
    capacity: int = 5000
    alpha_shrink: float = 100.0
    sigma_floor: float = 0.15
    min_samples_shrink: int = 50

    @classmethod
    def from_calibration(cls, d):
        return cls(sigma=d["SIGMA"], merge_threshold=d["MERGE_THRESHOLD"],
                   v_max=d.get("V_MAX", 0.5), capacity=d.get("DR_CAPACITY", 5000))


@dataclass
class MemoryCenter:
    z: np.ndarray
    v: float = 0.0
    w: float = 0.0
    n_updates: int = 0
    D_sum: float = 0.0
    D_sq_sum: float = 0.0
    mu_eff: float = 0.06
    context_id: int = -1
    birth_epoch: int = 0
    context_update_counts: Dict[int, int] = field(default_factory=dict)
    sigma: float = 0.58
    D_history: List[float] = field(default_factory=list)
    D_history_cross: List[float] = field(default_factory=list)
    was_ever_crystallized: bool = False
    crucible_verified: bool = False
    dissolution_log: List[Dict] = field(default_factory=list)

    def is_crystallized(self):
        return self.mu_eff < 0.06 - 1e-6

    def D_var_rolling(self):
        if len(self.D_history) < 25:
            return 0.0
        r = self.D_history[20:][-50:]
        return float(np.var(r)) if len(r) >= 5 else 0.0


class IBFEngine:
    """Universal IBF engine — discrete particle approximation of Postulate 1.

    The engine is domain-agnostic: it operates on a d-dimensional latent space
    supplied by a frozen encoder and a baseline evaluator. Reading C patch is
    applied: the agency channel gates on history sufficiency rather than on
    crystallisation, bringing it into parity with the value channel and matching
    Postulate 1's parallel-equation framing.
    """

    def __init__(self, params=None):
        self.p = params or IBFParams()
        self.value_centers, self.agency_centers = [], []
        self.current_epoch = 0
        self.current_context_id = -1
        self._epoch_D_vals = []
        self._merge_ratio = self.p.merge_threshold / self.p.sigma
        self._dynamic_sigma_floor = min(self.p.sigma_floor, self.p.sigma * 0.25)
        self._needle_threshold = self.p.sigma * 0.50
        self._sigma_agency = self.p.sigma_agency if self.p.sigma_agency else self.p.sigma
        self._D_running_sum = 0.0
        self._D_running_count = 0
        print(f"IBF: σ={self.p.sigma:.4f}, cap={self.p.capacity}, engine_version={ENGINE_VERSION}")

    def kernel_batch(self, z, centers):
        if not centers:
            return np.array([])
        za = np.array([c.z for c in centers])
        sq = np.sum((za - z[None, :]) ** 2, 1)
        return np.exp(-sq / (2 * np.array([c.sigma for c in centers]) ** 2))

    def _read_gate(self, c):
        if c.context_id == self.current_context_id:
            return 1.0
        return 1.0 if c.is_crystallized() and c.crucible_verified else 0.0

    def delta_R(self, z):
        if not self.value_centers:
            return 0.0
        K = self.kernel_batch(z, self.value_centers); t = 0.0
        for i, c in enumerate(self.value_centers):
            g = self._read_gate(c)
            if g > 0 and K[i] > self.p.activation_threshold:
                t += g * c.v * K[i]
        return t

    def delta_k(self, z):
        """Intensive responsiveness readout (foundation § 4.2, Eq. 8).

        Reading C patch: gate on history sufficiency, not crystallisation.
        """
        if not self.agency_centers:
            return 0.0
        K = self.kernel_batch(z, self.agency_centers); tw = sk = 0.0
        for i, c in enumerate(self.agency_centers):
            # PATCHED: history-sufficiency gate replaces crystallisation gate.
            if len(c.D_history) < self.p.n_agency_min:
                continue
            g = self._read_gate(c)
            if g > 0 and K[i] > self.p.activation_threshold:
                tw += g * c.w * K[i]; sk += g * K[i]
        return tw / sk if sk > 1e-6 else 0.0

    def k_eff(self, z):
        return max(self.p.k_min, self.p.k_0 + self.delta_k(z))

    def compute_D_and_update(self, board_before, board_after_own_move,
                             board_after_opponent, move_chosen=None,
                             move_opponent=None, R_imposed_override=None):
        z_before = RC_encode(board_before)
        z_chosen = (RC_encode_move(board_before, board_after_own_move, move_chosen)
                    if move_chosen is not None else RC_encode(board_after_own_move))
        R_f = RC_R_field(board_after_own_move)
        dR = self.delta_R(z_chosen)
        R_eff_ch = np.clip(1.0 - (R_f + dR), 0, 1)
        R_eff_imp = float(R_imposed_override) if R_imposed_override is not None else 0.5
        D = R_eff_imp - R_eff_ch
        self._D_running_count += 1; self._D_running_sum += D
        D -= self._D_running_sum / self._D_running_count
        self._epoch_D_vals.append(D)
        self._update_value(z_chosen, D)
        self._update_agency(z_before, D)
        return {"D": D, "R_eff_chosen": float(R_eff_ch), "dR_chosen": float(dR)}

    def _shrink(self, c):
        if len(c.D_history) >= self.p.min_samples_shrink:
            dv = c.D_var_rolling()
            c.sigma = min(c.sigma, max(self._dynamic_sigma_floor,
                          self.p.sigma / (1 + self.p.alpha_shrink * dv)))

    def _update_value(self, z, D):
        neg_D = -D; ctx = self.current_context_id
        for c in [c for c in self.value_centers if c.is_crystallized() and c.context_id != ctx]:
            kw = float(self.kernel_batch(z, [c])[0])
            if kw < self.p.activation_threshold:
                continue
            c.v = np.clip(c.v + self.p.eta_cryst * neg_D * kw, -self.p.v_max, self.p.v_max)
            c.n_updates += 1
            c.context_update_counts[ctx] = c.context_update_counts.get(ctx, 0) + 1
            c.D_history_cross.append(neg_D)
        sc = [c for c in self.value_centers if c.context_id == ctx]
        if sc:
            Ks = self.kernel_batch(z, sc); mx = float(np.max(Ks))
        else:
            Ks = np.array([]); mx = 0.0
        if mx < self.p.creation_threshold:
            if len(self.value_centers) < self.p.capacity:
                nc = MemoryCenter(
                    z=z.copy(),
                    v=np.clip(self.p.eta * neg_D, -self.p.v_max, self.p.v_max),
                    mu_eff=self.p.mu_base, context_id=ctx,
                    birth_epoch=self.current_epoch, sigma=self.p.sigma,
                )
                nc.n_updates = 1; nc.D_sum = neg_D; nc.D_sq_sum = neg_D ** 2
                nc.D_history.append(neg_D); nc.context_update_counts[ctx] = 1
                self.value_centers.append(nc)
            return
        for i, c in enumerate(sc):
            kw = float(Ks[i])
            if kw < self.p.activation_threshold:
                continue
            lr = self.p.eta_cryst if c.is_crystallized() else self.p.eta
            c.v = np.clip(c.v + lr * neg_D * kw, -self.p.v_max, self.p.v_max)
            c.n_updates += 1; c.D_sum += neg_D * kw; c.D_sq_sum += (neg_D * kw) ** 2
            c.context_update_counts[ctx] = c.context_update_counts.get(ctx, 0) + 1
            c.D_history.append(neg_D * kw); self._shrink(c)

    def _update_agency(self, z, D):
        """Responsiveness modification — parallel equation to δR (Postulate 1).

        Reading C patch: history-sufficiency gates (`len(c.D_history) >=
        n_agency_min`) replace the crystallisation gate, and a parallel
        learning rate `eta_k_cryst` matches `eta_cryst` for the value channel.
        """
        ctx = self.current_context_id
        for c in [c for c in self.agency_centers if c.is_crystallized() and c.context_id != ctx]:
            kw = float(self.kernel_batch(z, [c])[0])
            if kw < self.p.activation_threshold:
                continue
            c.n_updates += 1
            c.context_update_counts[ctx] = c.context_update_counts.get(ctx, 0) + 1
            c.D_history_cross.append(D)
        sc = [c for c in self.agency_centers if c.context_id == ctx]
        if sc:
            Ks = self.kernel_batch(z, sc); mx = float(np.max(Ks))
        else:
            Ks = np.array([]); mx = 0.0
        if mx < self.p.creation_threshold:
            if len(self.agency_centers) < self.p.capacity:
                nc = MemoryCenter(
                    z=z.copy(), mu_eff=self.p.mu_base, context_id=ctx,
                    birth_epoch=self.current_epoch, sigma=self._sigma_agency,
                )
                nc.n_updates = 1; nc.D_sum = D; nc.D_sq_sum = D ** 2
                nc.D_history.append(D); nc.context_update_counts[ctx] = 1
                self.agency_centers.append(nc)
            return
        for i, c in enumerate(sc):
            kw = float(Ks[i])
            if kw < self.p.activation_threshold:
                continue
            c.n_updates += 1; c.D_sum += D * kw; c.D_sq_sum += (D * kw) ** 2
            c.context_update_counts[ctx] = c.context_update_counts.get(ctx, 0) + 1
            c.D_history.append(D * kw)
            # PATCHED: history-sufficiency gate; parallel learning rates.
            if len(c.D_history) >= self.p.n_agency_min:
                dv = c.D_var_rolling()
                tw = np.clip(self.p.w_max * (1 - dv / self.p.w_dvar_threshold),
                             -self.p.w_max, self.p.w_max)
                lr = self.p.eta_k_cryst if c.is_crystallized() else self.p.eta_k
                c.w += lr * kw * (tw - c.w)
                c.w = np.clip(c.w, -self.p.w_max, self.p.w_max)
            self._shrink(c)

    def _crystallize(self):
        for pop in [self.value_centers, self.agency_centers]:
            for c in pop:
                if c.is_crystallized() or c.n_updates < self.p.crystallization_threshold:
                    continue
                if len(c.D_history) < 5:
                    continue
                if abs(float(np.mean(c.D_history[-50:]))) < self.p.convergence_threshold:
                    c.mu_eff = self.p.mu_crystallized
                    c.was_ever_crystallized = True

    def _crucible(self):
        nv = nd = 0
        for pop, uw in [(self.value_centers, False), (self.agency_centers, True)]:
            for c in pop:
                if not c.is_crystallized():
                    continue
                nc = len(c.D_history_cross)
                if nc < self.p.n_cross_min:
                    continue
                mu = float(np.mean(c.D_history_cross[-min(nc, 50):]))
                if not uw:
                    prod, thr = c.v * mu, self.p.reversal_threshold
                else:
                    prod = c.w * -abs(mu)
                    thr = self.p.reversal_threshold * (self.p.w_max / self.p.v_max)
                if prod < thr:
                    c.mu_eff = self.p.mu_base
                    c.crucible_verified = False
                    nd += 1
                else:
                    c.crucible_verified = True
                    nv += 1
        return nv, nd

    def reset_verifications(self):
        for c in self.value_centers + self.agency_centers:
            c.crucible_verified = False
            c.D_history_cross = []

    def _merge_pop(self, centers):
        if len(centers) < 2:
            return centers
        merged, new = set(), []
        for i in range(len(centers)):
            if i in merged:
                continue
            best = centers[i]
            for j in range(i + 1, len(centers)):
                if j in merged or centers[i].context_id != centers[j].context_id:
                    continue
                d = np.linalg.norm(centers[i].z - centers[j].z)
                ni = centers[i].sigma < self._needle_threshold
                nj = centers[j].sigma < self._needle_threshold
                thr = (self._merge_ratio * max(centers[i].sigma, centers[j].sigma) * 1.5
                       if ni and nj
                       else self._merge_ratio * min(centers[i].sigma, centers[j].sigma))
                if d < thr:
                    o = centers[j]
                    if o.n_updates > best.n_updates:
                        best, o = o, best
                    best.v = np.clip(best.v + o.v, -self.p.v_max, self.p.v_max)
                    best.w = np.clip(best.w + o.w, -self.p.w_max, self.p.w_max)
                    best.n_updates += o.n_updates
                    best.D_sum += o.D_sum; best.D_sq_sum += o.D_sq_sum
                    for k2, v2 in o.context_update_counts.items():
                        best.context_update_counts[k2] = best.context_update_counts.get(k2, 0) + v2
                    best.D_history.extend(o.D_history)
                    best.D_history_cross.extend(o.D_history_cross)
                    best.sigma = min(best.sigma, o.sigma)
                    if o.was_ever_crystallized:
                        best.was_ever_crystallized = True
                    merged.add(j)
            new.append(best)
        if len(new) > self.p.capacity:
            cr = [c for c in new if c.is_crystallized()]
            tr = sorted([c for c in new if not c.is_crystallized()],
                        key=lambda c: (abs(c.v) + abs(c.w)) * c.n_updates)
            nk = self.p.capacity - len(cr)
            new = cr + tr[-nk:] if nk > 0 else cr[:self.p.capacity]
        return new

    def end_epoch(self):
        for pop in [self.value_centers, self.agency_centers]:
            for c in pop:
                c.v *= (1 - c.mu_eff)
                c.w *= (1 - c.mu_eff)
        self._crystallize()
        nv, nd = self._crucible()
        self.value_centers = self._merge_pop(self.value_centers)
        self.agency_centers = self._merge_pop(self.agency_centers)
        D = self._epoch_D_vals; vs = [c.sigma for c in self.value_centers]
        diag = {
            "n_value_centers": len(self.value_centers),
            "n_value_crystallized": sum(1 for c in self.value_centers if c.is_crystallized()),
            "n_value_verified": sum(1 for c in self.value_centers
                                    if c.is_crystallized() and c.crucible_verified),
            "D_abs_mean": float(np.mean(np.abs(D))) if D else 0.0,
            "delta_R_max_abs": float(max((abs(c.v) for c in self.value_centers), default=0)),
            "n_dissolved": nd,
            "sigma_min": float(np.min(vs)) if vs else self.p.sigma,
        }
        self._epoch_D_vals = []
        self.current_epoch += 1
        return diag

    def set_context(self, ctx):
        self.current_context_id = ctx


# Shared JSON helper used across every artifact-writing cell.
def _jsonify(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    raise TypeError(f"Object of type {type(obj)} not JSON serializable")


def _write_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=_jsonify)


print(f"OK  IBFEngine ready  (ENGINE_VERSION={ENGINE_VERSION})")
print("    Reading C patch active:")
print("      - delta_k gates on len(c.D_history) >= n_agency_min, not crystallisation")
print("      - _update_agency uses parallel eta_k_cryst when crystallised")


## § S2 — Representation geometry calibration

**Objective.** Construct the 80-D proposition space and the 64-D agency space, then
calibrate the operating bandwidth σ to the geometrically prescribed value
(σ_operating ≈ 7.2621 on FI).

**Mapping to foundational concepts.**

- **Condition R / R′ (foundation § 3.5 — Discrete Convergence).** The mpnet encoder must
  produce a latent space in which the coherence function R̂ is C²-smooth with sufficient
  absolute magnitude. The 64-D PCA projection of all-mpnet-base-v2 sentence embeddings
  satisfies this empirically.
- **Condition A.** The discrete action set (4 multiple-choice answers) contains at least
  one action with a positive coherence increment. Verified at calibration time.
- **σ choice.** The kernel bandwidth σ is derived from latent geometry rather than grid
  search. The operating law is:

  ```
  σ_operating = min(
      d_pair  / sqrt(2 log(1 / ε_pair)),
      d_shell / sqrt(2 log(N_eff / ε_global))
  )
  ```

  with ε_pair = 0.05, ε_global = 5e-4, N_eff = 10000. The resulting σ ≈ 7.2621 is **the
  paper's canonical operating point** and is never re-tuned downstream.

**Frozen base model.** Mistral-7B-v0.1 (the LLM-extension flag in foundation § 5.4). The
base model's `R_base` (softmax over A/B/C/D answer tokens) supplies the baseline
coherence; IBF contributes only the local correction `δR`.


In [ ]:
# ════════════════════════════════════════════════════════════════
# § S2 — Representation geometry calibration
# Layer: setup
# Presupposes: S1 (engine)
# Artifacts: representation_geometry_calibration.json + .md
# Convergence-stop: n/a
# ════════════════════════════════════════════════════════════════
#
# Builds the 80-D proposition space (PCA 64 + subject 8 + answer 8) and
# calibrates the operating bandwidth σ from latent geometry. The 64-D
# agency space is the PCA output directly.
#
# This cell installs and loads sentence-transformers (mpnet-base-v2) and
# the frozen Mistral-7B base model. On a fresh pod this takes ~10-15 min
# the first time; subsequent runs hit the HuggingFace cache.
# ════════════════════════════════════════════════════════════════

import subprocess, sys

def _pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# Install dependencies. Idempotent — pip skips already-satisfied packages.
_pip_install("torch", "transformers", "datasets", "scikit-learn",
             "numpy", "accelerate", "sentence-transformers", "faiss-cpu",
             "scipy")

import math
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.spatial.distance import cdist
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer
import warnings
warnings.filterwarnings("ignore")

# ----- Frozen base model -----
model_name = "mistralai/Mistral-7B-v0.1"
N_CHOICES = 4
CHOICE_LABELS = ["A", "B", "C", "D"]
MAX_TOKEN_LEN = 256

print(f"Loading {model_name} ...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float16,
).to(DEVICE)
model.eval()
for pp in model.parameters():
    pp.requires_grad = False
HIDDEN_DIM = model.config.hidden_size
print(f"  ✓ Loaded: hidden={HIDDEN_DIM}")


def resolve_ids(tok):
    ids = {}
    for ch in CHOICE_LABELS:
        for cand in [ch, f" {ch}", f"\n{ch}"]:
            ts = tok.encode(cand, add_special_tokens=False)
            if len(ts) == 1:
                ids[ch] = ts[0]; break
        else:
            ts = tok.encode(ch, add_special_tokens=False); ids[ch] = ts[-1]
    return ids


CHOICE_TOKEN_IDS = resolve_ids(tokenizer)
CHOICE_IDS_ARRAY = [CHOICE_TOKEN_IDS[c] for c in CHOICE_LABELS]


@torch.no_grad()
def extract_base(prompts, bs=8):
    """Extract R_base_probs (over A/B/C/D) AND z_question (last-token hidden state)."""
    zs, ps = [], []
    for s in range(0, len(prompts), bs):
        b = prompts[s:s + bs]
        tokenizer.padding_side = "left"
        enc = tokenizer(b, return_tensors="pt", padding=True,
                        truncation=True,
                        max_length=min(512, MAX_TOKEN_LEN * 2)).to(DEVICE)
        out = model(**enc, output_hidden_states=True)
        zs.append(out.hidden_states[-1][:, -1, :].float().cpu().numpy())
        lg = out.logits[:, -1, :][:, CHOICE_IDS_ARRAY]
        ps.append(F.softmax(lg.float(), dim=-1).cpu().numpy())
    return np.concatenate(zs, axis=0), np.concatenate(ps, axis=0)


# ----- Representation geometry config (locked at paper's operating point) -----
Z_DIM = 64
SUBJECT_DIM = 8
ANSWER_DIM = 8
SUBJECT_SCALE = 25.0
ANSWER_SCALE = 25.0
Z_DIM_AUG = Z_DIM + SUBJECT_DIM + ANSWER_DIM  # 80D

CELL6_EPS_PAIR = 0.05
CELL6_EPS_GLOBAL = 5e-4
CELL6_N_EFF = 10000

print(f"Representation config: Z={Z_DIM}, +subj{SUBJECT_DIM}, +ans{ANSWER_DIM} = {Z_DIM_AUG}D")
print(f"σ calibration target: ε_pair={CELL6_EPS_PAIR}, ε_global={CELL6_EPS_GLOBAL}, N_eff={CELL6_N_EFF}")
print(f"Expected operating σ ≈ 7.2621 (paper canonical)")
print()
print("Full PCA fit + σ calibration runs after S3 builds the FI dataset.")
print("This split is structural: σ is calibrated on the FI proposition geometry,")
print("which only exists once S3 has materialised the phase data.")


## § S3 — Future Industries dataset + adapter + FAISS

**Objective.** Build the synthetic 1,000-employee organisation (*Future Industries*) used
across every claim section, define the four canonical lifecycle phases
(A_Onboarding → B_Initiative → C_Reorg → D_Turnover), extract `R_base` from the frozen
Mistral-7B, encode all propositions through mpnet, and complete σ calibration on the
resulting proposition geometry.

**Phase semantics (foundation § 3, Crucible / context-gated read access).**

| Phase | Role |
|---|---|
| A_Onboarding | Knowledge injection (new entities, new facts) |
| B_Initiative | New context, distinct domain (cross-context isolation test) |
| C_Reorg | Targeted revisions inside A's domain (truth-maintenance) |
| D_Turnover | New roles + retraction (final lifecycle stress) |

Each phase has train and test splits; for every test item there are 4 multiple-choice
answers. `R_base` is the softmax over the A/B/C/D answer tokens; IBF's δR adds a local
correction on top.

The cell consolidates four concerns from the original notebook:

1. FI data generation (existing cell 11).
2. Base-model R_base extraction (existing cell 13).
3. Sentence embeddings + 80-D proposition geometry + σ calibration (existing cells 15-16).
4. Propositional adapter + FAISS acceleration (existing cells 18, 20).

End of S3 leaves the global `ibf` engine ready for C1 to train.


In [ ]:
# ════════════════════════════════════════════════════════════════
# § S3 — Future Industries dataset + adapter + FAISS
#
# FI dataset section is VERBATIM from the original notebook's cell 11
# (CELL 3 in the original numbering). Adapter + FAISS sections are
# from v2's setup; they integrate cleanly with the dataset.
# ════════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════════
# CELL 3: Experimental scenario: Future Industries data generation
# ══════════════════════════════════════════════════════════════════

print("=" * 70)
print("  CELL 3: EXPERIMENTAL SCENARIO")
print("  10 templates × 7 categories, retraction targets pre-designated")
print("=" * 70)

N_CHOICES = 4
CHOICE_LABELS = ['A', 'B', 'C', 'D']
rng = random.Random(SEED)

# Retraction target designation uses its own seed - stable across SEED changes
RETRACTION_TARGET_SEED = 2026
FROZEN_HELDOUT_SEED = 2026  # Used later in Cell 5

# ══════════════════════════════════════════════════════════════
# COMPANY STRUCTURE - unchanged from original
# ══════════════════════════════════════════════════════════════

COMPANY_NAME = "Future Industries"

TEAM_STRUCTURE = {
    'Engineering': [
        'Frontend Engineering', 'Backend Engineering', 'Infrastructure',
        'Mobile Engineering', 'ML Engineering', 'Platform Engineering',
        'DevOps', 'Quality Assurance',
    ],
    'Product': [
        'Growth Product', 'Core Product', 'Enterprise Product',
        'Platform Product', 'Product Analytics',
    ],
    'Design': [
        'UX Design', 'UI Design', 'Design Research',
        'Brand Design', 'Design Systems',
    ],
    'Marketing': [
        'Brand Marketing', 'Content Marketing', 'Demand Generation',
        'Event Marketing', 'SEO', 'Social Media',
    ],
    'Sales': [
        'Enterprise Sales', 'Mid-Market Sales', 'SMB Sales',
        'Partnerships', 'Solutions Engineering',
    ],
    'Finance': [
        'Accounting', 'FP&A', 'Treasury', 'Tax', 'Procurement',
    ],
    'Legal': [
        'Corporate Law', 'Intellectual Property', 'Compliance',
        'Privacy', 'Employment Law',
    ],
    'Operations': [
        'IT Operations', 'Facilities', 'Supply Chain',
        'Business Operations', 'Program Management',
    ],
    'Data Science': [
        'Analytics', 'ML Research', 'Data Engineering',
        'Business Intelligence', 'Applied AI',
    ],
    'Security': [
        'Application Security', 'Infrastructure Security',
        'GRC', 'Identity & Access', 'Incident Response',
    ],
    'People': [
        'Recruiting', 'People Operations', 'Learning & Development',
        'Compensation & Benefits', 'HR Business Partners',
    ],
}

ALL_TEAMS = []
TEAM_TO_DEPT = {}
for dept, teams in TEAM_STRUCTURE.items():
    for team in teams:
        ALL_TEAMS.append(team)
        TEAM_TO_DEPT[team] = dept

PROJECTS = [
    'Phoenix', 'Atlas', 'Horizon', 'Meridian', 'Catalyst',
    'Prism', 'Vanguard', 'Beacon', 'Nexus', 'Keystone',
    'Orbit', 'Frontier', 'Mosaic', 'Compass', 'Pinnacle',
    'Helix', 'Apex', 'Zenith', 'Forge', 'Pulse',
    'Eclipse', 'Ember', 'Titan', 'Aurora', 'Summit',
    'Quantum', 'Nova', 'Spark', 'Odyssey', 'Vertex',
]

BUILDINGS = ['Building A', 'Building B', 'Building C', 'Building D']
FLOORS = ['Floor 1', 'Floor 2', 'Floor 3', 'Floor 4', 'Floor 5', 'Floor 6']
LOCATIONS = [f"{b}, {f}" for b in BUILDINGS for f in FLOORS]

CERTIFICATIONS = [
    'AWS Solutions Architect', 'AWS DevOps Engineer', 'Google Cloud Professional',
    'Azure Administrator', 'Azure Solutions Architect', 'Kubernetes Administrator',
    'PMP', 'Scrum Master', 'SAFe Agilist', 'PRINCE2 Practitioner',
    'CISSP', 'CISM', 'CompTIA Security+', 'CEH', 'OSCP',
    'CPA', 'CFA Level I', 'CFA Level II', 'Series 7', 'Series 63',
    'Six Sigma Green Belt', 'Six Sigma Black Belt', 'Lean Practitioner',
    'TOGAF', 'ITIL Foundation', 'ITIL Expert',
    'Google Analytics', 'HubSpot Inbound', 'Salesforce Administrator',
    'Tableau Desktop Specialist', 'Power BI Analyst', 'dbt Analytics Engineer',
    'TensorFlow Developer', 'PyTorch Certified', 'MLflow Specialist',
    'Figma Professional', 'Adobe XD Expert', 'WCAG Accessibility',
    'SOC 2 Auditor', 'ISO 27001 Lead Implementer',
]

COMMITTEES = [
    'Ethics Committee', 'Diversity & Inclusion Council', 'Innovation Board',
    'Safety Committee', 'Sustainability Council', 'Culture Committee',
    'Technical Standards Board', 'Architecture Review Board',
    'Data Governance Council', 'Privacy Review Board',
    'Open Source Committee', 'Patent Review Committee',
    'Employee Wellness Committee', 'Social Impact Council',
    'New Hire Mentorship Board', 'Leadership Development Council',
    'Product Advisory Board', 'Customer Advisory Council',
    'Risk Management Committee', 'Budget Review Board',
    'Vendor Selection Committee', 'Workplace Design Committee',
    'AI Ethics Board', 'Accessibility Council', 'Security Review Board',
]

# ══════════════════════════════════════════════════════════════
# NAME GENERATION - unchanged
# ══════════════════════════════════════════════════════════════

FIRST_NAMES = [
    'Wei', 'Yuki', 'Hiro', 'Mei', 'Jin', 'Sora', 'Kai', 'Ren',
    'Yuna', 'Tao', 'Hana', 'Koji', 'Aiko', 'Ryu', 'Min', 'Suki',
    'Kenji', 'Nari', 'Jia', 'Haruto', 'Sakura', 'Chen', 'Linh', 'Dae',
    'Priya', 'Arjun', 'Ananya', 'Rohan', 'Kavya', 'Vikram', 'Neha', 'Arun',
    'Divya', 'Raj', 'Meera', 'Sanjay', 'Isha', 'Nikhil', 'Pooja', 'Amit',
    'Lakshmi', 'Rahul', 'Shreya', 'Kiran', 'Ravi', 'Nisha', 'Deepak', 'Anjali',
    'Elena', 'Marco', 'Sofia', 'Lars', 'Ingrid', 'Pierre', 'Clara', 'Anton',
    'Marta', 'Luca', 'Freya', 'Emil', 'Nina', 'Oscar', 'Elsa', 'Hugo',
    'Katja', 'Stefan', 'Lena', 'Franz', 'Ada', 'Nils', 'Vera', 'Leo',
    'Maya', 'Alex', 'Jordan', 'Taylor', 'Morgan', 'Riley', 'Quinn', 'Drew',
    'Carmen', 'Diego', 'Lucia', 'Rafael', 'Valentina', 'Mateo', 'Camila', 'Andre',
    'Olivia', 'Ethan', 'Zoe', 'Noah', 'Ava', 'Liam', 'Emma', 'Milo',
    'Amara', 'Kofi', 'Zara', 'Omar', 'Fatima', 'Idris', 'Aisha', 'Kwame',
    'Nadia', 'Tariq', 'Leila', 'Youssef', 'Amina', 'Hassan', 'Safiya', 'Ibrahim',
    'Chioma', 'Tendai', 'Nia', 'Jelani', 'Adaeze', 'Malik', 'Sanaa', 'Emeka',
]

LAST_NAMES = [
    'Chen', 'Kim', 'Tanaka', 'Nguyen', 'Li', 'Sato', 'Park', 'Wong',
    'Yamamoto', 'Zhao', 'Nakamura', 'Choi', 'Suzuki', 'Huang', 'Watanabe', 'Lin',
    'Patel', 'Shah', 'Kumar', 'Singh', 'Gupta', 'Sharma', 'Reddy', 'Mehta',
    'Kapoor', 'Chatterjee', 'Nair', 'Iyer', 'Malhotra', 'Bose', 'Rao', 'Desai',
    'Mueller', 'Johansson', 'Bernard', 'Rossi', 'Andersson', 'Weber', 'Dubois', 'Fischer',
    'Larsen', 'Moretti', 'Berg', 'Santos', 'Kowalski', 'Novak', 'Petrov', 'Brennan',
    'Rivera', 'Campbell', 'Mitchell', 'Torres', 'Brooks', 'Foster', 'Reed', 'Hayes',
    'Sullivan', 'Reyes', 'Griffin', 'Flores', 'Cole', 'Barnes', 'Cruz', 'Wells',
    'Okafor', 'Al-Rashid', 'Mensah', 'Diallo', 'El-Amin', 'Adeyemi', 'Toure', 'Osei',
    'Nkosi', 'Abadi', 'Mwangi', 'Khoury', 'Achebe', 'Saleh', 'Okoro', 'Bello',
]

# ══════════════════════════════════════════════════════════════
# ORG GENERATOR - unchanged
# ══════════════════════════════════════════════════════════════

N_EMPLOYEES = 1000

def generate_org(n_employees, rng_local):
    first_pool = list(FIRST_NAMES); last_pool = list(LAST_NAMES)
    rng_local.shuffle(first_pool); rng_local.shuffle(last_pool)
    all_names, name_set = [], set()
    for f in first_pool:
        for l in last_pool:
            name = f"{f} {l}"
            if name not in name_set:
                name_set.add(name); all_names.append(name)
            if len(all_names) >= n_employees: break
        if len(all_names) >= n_employees: break
    all_names = all_names[:n_employees]

    employees = []
    name_idx = 0

    employees.append({'name': all_names[name_idx], 'level': 0,
        'team': 'Executive Office', 'department': 'Executive',
        'manager': None, 'location': rng_local.choice(LOCATIONS)})
    name_idx += 1

    vps = {}
    for dept in TEAM_STRUCTURE:
        vp = {'name': all_names[name_idx], 'level': 1,
            'team': f'{dept} Leadership', 'department': dept,
            'manager': employees[0]['name'],
            'location': rng_local.choice(LOCATIONS)}
        employees.append(vp); vps[dept] = vp; name_idx += 1

    directors = {t: None for t in ALL_TEAMS}
    for team in ALL_TEAMS:
        if name_idx >= n_employees: break
        dept = TEAM_TO_DEPT[team]
        d = {'name': all_names[name_idx], 'level': 2, 'team': team,
            'department': dept, 'manager': vps[dept]['name'],
            'location': rng_local.choice(LOCATIONS)}
        employees.append(d); directors[team] = d; name_idx += 1

    managers = {t: [] for t in ALL_TEAMS}
    for team in ALL_TEAMS:
        for _ in range(rng_local.choice([1, 2])):
            if name_idx >= n_employees: break
            dept = TEAM_TO_DEPT[team]
            boss = directors[team] if directors[team] else vps[dept]
            m = {'name': all_names[name_idx], 'level': 3, 'team': team,
                'department': dept, 'manager': boss['name'],
                'location': rng_local.choice(LOCATIONS)}
            employees.append(m); managers[team].append(m); name_idx += 1

    team_cycle = list(ALL_TEAMS); rng_local.shuffle(team_cycle); tc_idx = 0
    while name_idx < n_employees:
        team = team_cycle[tc_idx % len(team_cycle)]; tc_idx += 1
        dept = TEAM_TO_DEPT[team]
        mgr_pool = managers[team] if managers[team] else (
            [directors[team]] if directors[team] else [vps[dept]])
        emp = {'name': all_names[name_idx],
            'level': rng_local.choice([4, 5]), 'team': team,
            'department': dept, 'manager': rng_local.choice(mgr_pool)['name'],
            'location': rng_local.choice(LOCATIONS)}
        employees.append(emp); name_idx += 1

    for emp in employees:
        emp['project'] = rng_local.choice(PROJECTS)

    senior_by_dept = {}
    for e in employees:
        if e['level'] in [2, 3]:
            senior_by_dept.setdefault(e['department'], []).append(e['name'])
    for emp in employees:
        pool = [n for n in senior_by_dept.get(emp['department'], [])
                if n != emp['name']]
        if not pool:
            pool = [n for n in senior_by_dept.get('Engineering', [])
                    if n != emp['name']]
        emp['mentor'] = rng_local.choice(pool) if pool else employees[0]['name']

    for emp in employees:
        emp['certification'] = rng_local.choice(CERTIFICATIONS)
        emp['committee'] = rng_local.choice(COMMITTEES)

    return employees

employees = generate_org(N_EMPLOYEES, rng)
emp_by_name = {e['name']: e for e in employees}

for e in employees:
    if e['manager'] is not None:
        assert e['manager'] in emp_by_name, f"{e['name']}'s manager not found"

# ══════════════════════════════════════════════════════════════
# ANSWER POOLS
# ══════════════════════════════════════════════════════════════

pools = {
    'team': sorted(set(e['team'] for e in employees)),
    'manager': sorted(set(e['manager'] for e in employees if e['manager'])),
    'project': PROJECTS,
    'mentor': sorted(set(e['mentor'] for e in employees)),
    'location': LOCATIONS,
    'certification': CERTIFICATIONS,
    'committee': COMMITTEES,
}

print(f"\n  Company: {COMPANY_NAME}, {N_EMPLOYEES} employees")
print(f"  Answer pool sizes (all must be ≥20):")
for cat, pool in pools.items():
    ok = "✓" if len(pool) >= 20 else "✗ FAIL"
    print(f"    {cat:<15s}: {len(pool):>3d}  {ok}")
    assert len(pool) >= 20, f"Pool {cat} has only {len(pool)} entries"

# ══════════════════════════════════════════════════════════════
# TEMPLATES - 10 train + 5 test per category
# Composition: 4 original + 2 passive + 2 embedded-clause + 2 fronted-modifier
# ══════════════════════════════════════════════════════════════

TEMPLATES = {
    'team': {
        'train': [
            # Original (baseline syntactic forms)
            "{s} is on the {c} team at Future Industries",
            "{s} works in {c} at Future Industries",
            "At Future Industries, {s} is part of {c}",
            "The team of {s} at Future Industries is {c}",
            # Passive voice
            "{s} has been assigned to {c} at Future Industries",
            "At Future Industries, {s} was placed on the {c} team",
            # Embedded clauses
            "It is known that {s}, who works at Future Industries, belongs to {c}",
            "Records indicate that {s} is a member of the {c} team at Future Industries",
            # Fronted modifiers
            "Within Future Industries's organizational structure, {s} sits on {c}",
            "Among the teams at Future Industries, {c} is where {s} works",
        ],
        'test': [
            "What team does {s} work on at Future Industries?",
            "Which team is {s} part of at Future Industries?",
            "At Future Industries, what team is {s} on?",
            "Which of Future Industries's teams has {s} as a member?",
            "On which team has {s} been placed at Future Industries?",
        ],
    },
    'manager': {
        'train': [
            "{s} reports to {c} at Future Industries",
            "{s}'s manager at Future Industries is {c}",
            "The direct supervisor of {s} is {c}",
            "{s} works under {c} at Future Industries",
            "{s} is managed by {c} at Future Industries",
            "At Future Industries, {s} is supervised by {c}",
            "At Future Industries, where reporting lines are formal, {s} reports to {c}",
            "It is documented that {s}'s direct report at Future Industries is {c}",
            "According to Future Industries's org chart, {s} reports to {c}",
            "On the Future Industries reporting structure, above {s} sits {c}",
        ],
        'test': [
            "Who does {s} report to at Future Industries?",
            "Who is {s}'s manager at Future Industries?",
            "Who supervises {s} at Future Industries?",
            "By whom is {s} managed at Future Industries?",
            "According to Future Industries's org chart, who is {s}'s manager?",
        ],
    },
    'project': {
        'train': [
            "{s} is assigned to Project {c}",
            "{s} works on Project {c} at Future Industries",
            "The primary project of {s} is {c}",
            "Project {c} has {s} as a team member",
            "{s} has been assigned to Project {c}",
            "At Future Industries, Project {c} is staffed by {s}",
            "At Future Industries, the project that {s} contributes to is {c}",
            "Records show that {s}, an employee at Future Industries, works on Project {c}",
            "Among Future Industries's active projects, {c} is where {s} is assigned",
            "In Future Industries's project portfolio, {s} contributes to {c}",
        ],
        'test': [
            "What project is {s} working on at Future Industries?",
            "Which project is {s} assigned to?",
            "What is {s}'s primary project at Future Industries?",
            "To which of Future Industries's projects has {s} been assigned?",
            "Among Future Industries's projects, which one is {s} working on?",
        ],
    },
    'mentor': {
        'train': [
            "{s}'s mentor at Future Industries is {c}",
            "{s} is mentored by {c}",
            "The assigned mentor of {s} is {c}",
            "{c} mentors {s} at Future Industries",
            "At Future Industries, {s} has been paired with {c} for mentorship",
            "{s} is guided professionally by {c} at Future Industries",
            "At Future Industries, the mentor who supports {s} is {c}",
            "It has been arranged that {s}, a Future Industries employee, receives mentorship from {c}",
            "Within Future Industries's mentorship program, {s} is matched with {c}",
            "Under Future Industries's mentorship arrangement, {c} guides {s}",
        ],
        'test': [
            "Who is {s}'s mentor at Future Industries?",
            "Who mentors {s} at Future Industries?",
            "Who is assigned as {s}'s mentor?",
            "By whom is {s} mentored at Future Industries?",
            "Under Future Industries's mentorship program, who guides {s}?",
        ],
    },
    'location': {
        'train': [
            "{s} sits in {c} at Future Industries",
            "{s}'s desk is in {c}",
            "The workspace of {s} is {c}",
            "{s} works from {c} at Future Industries",
            "{s} has been seated in {c} at Future Industries",
            "At Future Industries, {s} is located in {c}",
            "At Future Industries, the office where {s} works is {c}",
            "Facilities records show that {s}, an employee at Future Industries, occupies {c}",
            "Within Future Industries's offices, {c} is where {s} sits",
            "Across Future Industries's campus, {s}'s assigned workspace is {c}",
        ],
        'test': [
            "Where does {s} sit at Future Industries?",
            "What is {s}'s workspace location at Future Industries?",
            "In which building and floor is {s} located?",
            "At Future Industries, where has {s} been seated?",
            "According to Future Industries's facilities records, where does {s} work?",
        ],
    },
    'certification': {
        'train': [
            "{s} holds the {c} certification",
            "{s} is certified in {c}",
            "The professional certification of {s} is {c}",
            "{s} completed the {c} program",
            "The {c} certification has been earned by {s}",
            "At Future Industries, {s} was awarded the {c} credential",
            "At Future Industries, the certification that {s} holds is {c}",
            "It is on record that {s}, a Future Industries employee, has completed {c}",
            "Among {s}'s professional credentials, {c} is the primary one",
            "Within Future Industries's training records, {s} is listed as certified in {c}",
        ],
        'test': [
            "What certification does {s} hold?",
            "Which certification has {s} completed?",
            "What is {s}'s professional certification?",
            "Which credential has been awarded to {s} at Future Industries?",
            "Among {s}'s certifications, which one is on record?",
        ],
    },
    'committee': {
        'train': [
            "{s} serves on the {c}",
            "{s} is a member of the {c}",
            "The committee assignment of {s} is the {c}",
            "{s} participates in the {c}",
            "{s} has been appointed to the {c} at Future Industries",
            "At Future Industries, {s} is seated on the {c}",
            "At Future Industries, the committee on which {s} serves is the {c}",
            "Records confirm that {s}, a Future Industries employee, sits on the {c}",
            "Among Future Industries's governance bodies, the {c} includes {s}",
            "Within Future Industries's committee structure, {s} is assigned to the {c}",
        ],
        'test': [
            "What committee does {s} serve on?",
            "Which committee is {s} a member of?",
            "What is {s}'s committee assignment?",
            "To which committee has {s} been appointed at Future Industries?",
            "Within Future Industries's governance structure, which committee includes {s}?",
        ],
    },
}

# ── Sanity-check template counts ──
N_TRAIN_TEMPLATES = 10
N_TEST_TEMPLATES = 5
for cat, tmpl in TEMPLATES.items():
    assert len(tmpl['train']) == N_TRAIN_TEMPLATES, \
        f"Category {cat} has {len(tmpl['train'])} train templates, expected {N_TRAIN_TEMPLATES}"
    assert len(tmpl['test']) == N_TEST_TEMPLATES, \
        f"Category {cat} has {len(tmpl['test'])} test templates, expected {N_TEST_TEMPLATES}"
    for t in tmpl['train']:
        assert '{s}' in t and '{c}' in t, f"Train template missing placeholder: {t!r}"
    for t in tmpl['test']:
        assert '{s}' in t, f"Test template missing {{s}}: {t!r}"
        assert '{c}' not in t, f"Test template should not contain {{c}}: {t!r}"
print(f"\n  ✓ Template sanity check passed (10 train + 5 test per category × 7 categories)")

# ══════════════════════════════════════════════════════════════
# P.2 - TOKENIZER HEADROOM CHECK
# Measure max tokenized length of any formatted train template.
# Set MAX_TOKEN_LEN accordingly for Cell 4 extract_hidden/extract_base.
# ══════════════════════════════════════════════════════════════

print(f"\n  P.2 - Tokenizer headroom check...")
from transformers import AutoTokenizer as _AT
_tok_probe = _AT.from_pretrained(model_name)

# Worst-case: longest subject name × longest answer × longest template
_longest_subj = max((e['name'] for e in employees), key=len)
_longest_answers = {
    'team': max(pools['team'], key=len),
    'manager': max(pools['manager'], key=len),
    'project': max(PROJECTS, key=len),
    'mentor': max(pools['mentor'], key=len),
    'location': max(LOCATIONS, key=len),
    'certification': max(CERTIFICATIONS, key=len),
    'committee': max(COMMITTEES, key=len),
}

_max_tokens_seen = 0
_longest_prompt_sample = ""
for cat, tmpl in TEMPLATES.items():
    ans = _longest_answers[cat]
    for t in tmpl['train']:
        q = t.format(s=_longest_subj, c=ans)
        # Build a realistic base_prompt: question + ABCD choices block
        sample_choices = [ans] * 4
        base = (f"Question: {q}\nChoices:\n"
                f"A) {sample_choices[0]}\nB) {sample_choices[1]}\n"
                f"C) {sample_choices[2]}\nD) {sample_choices[3]}\nAnswer:")
        n_tok = len(_tok_probe.encode(base, add_special_tokens=True))
        if n_tok > _max_tokens_seen:
            _max_tokens_seen = n_tok
            _longest_prompt_sample = base[:100] + "..."

# Also check propositional prompts ("Question: q\nAnswer: choice")
for cat, tmpl in TEMPLATES.items():
    ans = _longest_answers[cat]
    for t in tmpl['train']:
        q = t.format(s=_longest_subj, c=ans)
        prop = f"Question: {q}\nAnswer: {ans}"
        n_tok = len(_tok_probe.encode(prop, add_special_tokens=True))
        if n_tok > _max_tokens_seen:
            _max_tokens_seen = n_tok

# Pick max_length: use 384 if anything exceeds 256, else keep 256
if _max_tokens_seen > 256:
    MAX_TOKEN_LEN = 384
else:
    MAX_TOKEN_LEN = 256

print(f"    Max tokens observed: {_max_tokens_seen}")
print(f"    MAX_TOKEN_LEN set to: {MAX_TOKEN_LEN}")
assert _max_tokens_seen <= MAX_TOKEN_LEN, \
    f"Max tokens {_max_tokens_seen} exceeds MAX_TOKEN_LEN {MAX_TOKEN_LEN}"
del _tok_probe

# ══════════════════════════════════════════════════════════════
# ITEM GENERATION - unchanged logic, just picks up 10 templates
# ══════════════════════════════════════════════════════════════

def make_company_items(subjects, answers, category, answer_pool, rng_local):
    tmpl = TEMPLATES[category]
    items_train, items_test = [], []
    for subj, ans in zip(subjects, answers):
        dists = [a for a in answer_pool if a != ans]
        if len(dists) < 3: continue
        chosen_dists = rng_local.sample(dists, 3)
        choices = [ans] + chosen_dists
        rng_local.shuffle(choices)
        label = choices.index(ans)
        common = {'choices': choices, 'label': label, 'subject': subj,
                  'answer': ans, 'category': category}
        def make_base(q, ch):
            return (f"Question: {q}\nChoices:\n"
                    f"A) {ch[0]}\nB) {ch[1]}\nC) {ch[2]}\nD) {ch[3]}\nAnswer:")
        for t_tmpl in tmpl['train']:
            q = t_tmpl.format(s=subj, c=ans)
            items_train.append({**common, 'base_prompt': make_base(q, choices),
                                'question': q})
        q_test = rng_local.choice(tmpl['test']).format(s=subj, c=ans)
        items_test.append({**common, 'base_prompt': make_base(q_test, choices),
                           'question': q_test})
    return items_train, items_test

# ══════════════════════════════════════════════════════════════
# ACT 1 (PHASE A): ONBOARDING
# ══════════════════════════════════════════════════════════════

non_exec = [e for e in employees if e['level'] > 0 and e['manager'] is not None]
rng.shuffle(non_exec)
phase_a_emps = non_exec[:200]
subjs = [e['name'] for e in phase_a_emps]

print(f"\n  ═══ ACT 1: ONBOARDING (Phase A) - 200 employees × 5 cats ═══")
a_train, a_test = [], []

categories_a = [
    ('team',     [e['team'] for e in phase_a_emps],     pools['team']),
    ('manager',  [e['manager'] for e in phase_a_emps],  pools['manager']),
    ('project',  [e['project'] for e in phase_a_emps],  pools['project']),
    ('mentor',   [e['mentor'] for e in phase_a_emps],   pools['mentor']),
    ('location', [e['location'] for e in phase_a_emps], pools['location']),
]

for cat_name, answers, pool in categories_a:
    tr, te = make_company_items(subjs, answers, cat_name, pool, rng)
    a_train.extend(tr); a_test.extend(te)
    print(f"    {cat_name:<15s}: {len(te):>3d} facts × {N_TRAIN_TEMPLATES} = "
          f"{len(tr):>5d} train  (pool: {len(pool)})")

print(f"    {'TOTAL':<15s}: {len(a_test):>3d} facts,      {len(a_train):>5d} train items")

# ══════════════════════════════════════════════════════════════
# P.6a - RETRACTION TARGET DESIGNATION
# 50 targets pre-selected from the 200 Phase A employees.
# Uses dedicated seed, stable across SEED changes for main pipeline.
# ══════════════════════════════════════════════════════════════

print(f"\n  P.6a - Designating retraction targets...")
_tgt_rng = random.Random(RETRACTION_TARGET_SEED)
_pa_names = [e['name'] for e in phase_a_emps]
_target_indices = _tgt_rng.sample(range(len(_pa_names)), 50)
RETRACTION_TARGETS = sorted([_pa_names[i] for i in _target_indices])
RETRACTION_TARGETS_SET = set(RETRACTION_TARGETS)

# Sanity checks
assert len(RETRACTION_TARGETS) == 50, \
    f"Expected 50 targets, got {len(RETRACTION_TARGETS)}"
assert all(n in _pa_names for n in RETRACTION_TARGETS), \
    "Some retraction targets are not in Phase A population"
print(f"    ✓ {len(RETRACTION_TARGETS)} retraction targets designated")
print(f"    Seed: {RETRACTION_TARGET_SEED} (independent of main SEED={SEED})")
print(f"    First 3 targets: {RETRACTION_TARGETS[:3]}")

# ══════════════════════════════════════════════════════════════
# ACT 2 (PHASE B): NEW INITIATIVE
# ══════════════════════════════════════════════════════════════

print(f"\n  ═══ ACT 2: NEW INITIATIVE (Phase B) - 200 employees × 2 cats ═══")
b_train, b_test = [], []

categories_b = [
    ('certification', [e['certification'] for e in phase_a_emps], pools['certification']),
    ('committee',     [e['committee'] for e in phase_a_emps],     pools['committee']),
]

for cat_name, answers, pool in categories_b:
    tr, te = make_company_items(subjs, answers, cat_name, pool, rng)
    b_train.extend(tr); b_test.extend(te)
    print(f"    {cat_name:<15s}: {len(te):>3d} facts × {N_TRAIN_TEMPLATES} = "
          f"{len(tr):>5d} train  (pool: {len(pool)})")

print(f"    {'TOTAL':<15s}: {len(b_test):>3d} facts,      {len(b_train):>5d} train items")

# ══════════════════════════════════════════════════════════════
# ACT 3 (PHASE C): QUARTERLY REORG
# ══════════════════════════════════════════════════════════════

print(f"\n  ═══ ACT 3: QUARTERLY REORG (Phase C) - 150 counterfactuals ═══")
c_train, c_test = [], []

mgr_train = [t for t in a_train if t['category'] == 'manager']
mgr_test = [t for t in a_test if t['category'] == 'manager']
n_mgr_reorg = min(100, len(mgr_test))

for fact_i in range(n_mgr_reorg):
    orig_te = mgr_test[fact_i].copy()
    old_label = orig_te['label']
    choices = list(orig_te['choices'])
    new_label = (old_label + 1) % N_CHOICES
    new_answer = choices[new_label]
    cf_te = {**orig_te, 'label': new_label, 'answer': new_answer,
             'original_answer': orig_te['answer'], 'counterfactual': True}
    c_test.append(cf_te)
    for t_idx in range(N_TRAIN_TEMPLATES):
        train_i = fact_i * N_TRAIN_TEMPLATES + t_idx
        if train_i >= len(mgr_train): break
        orig_tr = mgr_train[train_i].copy()
        cf_tr = {**orig_tr, 'label': new_label, 'answer': new_answer,
                 'original_answer': orig_tr['answer'], 'counterfactual': True}
        c_train.append(cf_tr)

prj_train = [t for t in a_train if t['category'] == 'project']
prj_test = [t for t in a_test if t['category'] == 'project']
n_prj_reorg = min(50, len(prj_test))

for fact_i in range(n_prj_reorg):
    orig_te = prj_test[fact_i].copy()
    old_label = orig_te['label']
    choices = list(orig_te['choices'])
    new_label = (old_label + 1) % N_CHOICES
    new_answer = choices[new_label]
    cf_te = {**orig_te, 'label': new_label, 'answer': new_answer,
             'original_answer': orig_te['answer'], 'counterfactual': True}
    c_test.append(cf_te)
    for t_idx in range(N_TRAIN_TEMPLATES):
        train_i = fact_i * N_TRAIN_TEMPLATES + t_idx
        if train_i >= len(prj_train): break
        orig_tr = prj_train[train_i].copy()
        cf_tr = {**orig_tr, 'label': new_label, 'answer': new_answer,
                 'original_answer': orig_tr['answer'], 'counterfactual': True}
        c_train.append(cf_tr)

print(f"    Manager changes:  {n_mgr_reorg}")
print(f"    Project changes:  {n_prj_reorg}")
print(f"    {'TOTAL':<15s}: {len(c_test):>3d} changes,    {len(c_train):>5d} train items")

# ══════════════════════════════════════════════════════════════
# ACT 4 (PHASE D): TURNOVER
# ══════════════════════════════════════════════════════════════

print(f"\n  ═══ ACT 4: TURNOVER (Phase D) - 30 out, 30 in ═══")
d_train, d_test = [], []

departing_emps = phase_a_emps[:30]
remaining_emps = phase_a_emps[30:]
departing_names = set(e['name'] for e in departing_emps)
remaining_a_test = [t for t in a_test if t['subject'] not in departing_names]
departed_a_test = [t for t in a_test if t['subject'] in departing_names]

print(f"    Departing: {len(departing_emps)} employees "
      f"({len(departed_a_test)} facts retired)")
print(f"    Remaining: {len(remaining_emps)} employees "
      f"({len(remaining_a_test)} facts should survive)")

new_hire_pool = [e for e in non_exec if e not in phase_a_emps]
rng.shuffle(new_hire_pool)
new_hires = new_hire_pool[:30]
new_hire_subjs = [e['name'] for e in new_hires]

categories_d = [
    ('team',    [e['team'] for e in new_hires],    pools['team']),
    ('manager', [e['manager'] for e in new_hires], pools['manager']),
    ('project', [e['project'] for e in new_hires], pools['project']),
]

for cat_name, answers, pool in categories_d:
    tr, te = make_company_items(new_hire_subjs, answers, cat_name, pool, rng)
    d_train.extend(tr); d_test.extend(te)
    print(f"    New hire {cat_name:<15s}: {len(te):>3d} facts × "
          f"{N_TRAIN_TEMPLATES} = {len(tr):>4d} train")

print(f"    New hire TOTAL:    {len(d_test):>3d} facts,      "
      f"{len(d_train):>4d} train items")

phase_d_meta = {
    'departing_names': list(departing_names),
    'remaining_a_test': remaining_a_test,
    'departed_a_test': departed_a_test,
    'new_hire_names': [e['name'] for e in new_hires],
}

PHASE_NAMES = ['A_Onboarding', 'B_Initiative', 'C_Reorg', 'D_Turnover']

phase_data = {
    'A_Onboarding': {'train': a_train, 'test': a_test},
    'B_Initiative': {'train': b_train, 'test': b_test},
    'C_Reorg':      {'train': c_train, 'test': c_test},
    'D_Turnover':   {'train': d_train, 'test': d_test},
}

# ══════════════════════════════════════════════════════════════
# FINAL SANITY CHECK - hard assertions on expected counts
# ══════════════════════════════════════════════════════════════

EXPECTED_COUNTS = {
    'A_Onboarding': {'train': 10000, 'test': 1000},   # 200 × 5 × 10 / 200 × 5
    'B_Initiative': {'train': 4000,  'test': 400},    # 200 × 2 × 10 / 200 × 2
    'C_Reorg':      {'train': 1500,  'test': 150},    # 150 × 10 / 150
    'D_Turnover':   {'train': 900,   'test': 90},     # 30 × 3 × 10 / 30 × 3
}

total_train = sum(len(phase_data[pn]['train']) for pn in PHASE_NAMES)
total_test = sum(len(phase_data[pn]['test']) for pn in PHASE_NAMES)

print(f"\n  ══════════════════════════════════════════════════")
print(f"  SUMMARY (with assertion-based count check)")
print(f"  ══════════════════════════════════════════════════")
for pn in PHASE_NAMES:
    tr = phase_data[pn]['train']; te = phase_data[pn]['test']
    cats = sorted(set(r['category'] for r in tr))
    exp = EXPECTED_COUNTS[pn]
    tr_ok = "✓" if len(tr) == exp['train'] else f"✗ expected {exp['train']}"
    te_ok = "✓" if len(te) == exp['test'] else f"✗ expected {exp['test']}"
    print(f"    {pn}: {len(tr):>5d} train {tr_ok}, {len(te):>4d} test {te_ok}  cats={cats}")
    assert len(tr) == exp['train'], \
        f"{pn} train count: got {len(tr)}, expected {exp['train']}"
    assert len(te) == exp['test'], \
        f"{pn} test count: got {len(te)}, expected {exp['test']}"
print(f"    TOTAL:  {total_train:>5d} train (expected 16,400), "
      f"{total_test:>4d} test (expected 1,640)")
assert total_train == 16400, f"Total train: got {total_train}, expected 16400"
assert total_test == 1640, f"Total test: got {total_test}, expected 1640"

print(f"\n  Design constraint check:")
for cat, pool in pools.items():
    used_in = []
    for pn in PHASE_NAMES:
        if any(r['category'] == cat for r in phase_data[pn]['train']):
            used_in.append(pn)
    if used_in:
        print(f"    {cat:<15s}: pool={len(pool):>3d}  used in {used_in}  "
              f"{'✓' if len(pool) >= 20 else '✗ BELOW 20'}")

# Verify retraction targets are in Phase A and Cell 5 can compute matchings
targets_in_a = sum(1 for t in a_test
                   if t['subject'] in RETRACTION_TARGETS_SET
                   and t['category'] == 'manager')
print(f"\n  Retraction target verification:")
print(f"    {len(RETRACTION_TARGETS)} targets × manager category → "
      f"{targets_in_a} manager facts available (expected 50)")
assert targets_in_a == 50, \
    f"Expected 50 target manager facts, got {targets_in_a}"

print(f"\n{'=' * 70}")
print(f"  ✓ {COMPANY_NAME}: {N_EMPLOYEES} employees, {total_test} unique facts")
print(f"  ✓ 10 train + 5 test templates per category")
print(f"  ✓ MAX_TOKEN_LEN = {MAX_TOKEN_LEN}")
print(f"  ✓ 50 retraction targets pre-designated (seed {RETRACTION_TARGET_SEED})")
print(f"  ✓ All count assertions passed (16,400 train / 1,640 test)")
print(f"  ✓ Ready for Cell 4 (extraction)")
print(f"{'=' * 70}")


# ══════════════════════════════════════════════════════════════════
# § S3 (continued) — Token IDs + base extraction + mpnet embeddings
#                    + 80-D PCA augmentation + σ calibration
#
# Sub-blocks below are ported from v1 cells 13, 15, 16 VERBATIM, with
# two non-content edits:
#   - duplicate Mistral re-load at the top of v1 cell 13 elided
#     (v2 S2 already loads `model`, `tokenizer`, `extract_base`)
#   - `!pip install sentence-transformers -q` magic in v1 cell 15
#     elided (v2 S2 already installs sentence-transformers)
#
# Produces the S3 contract:
#   precomputed[k]["z_question", "z_choices", "R_base_probs", "labels"]
#   SIGMA_PROP, SIGMA_AGENCY, MERGE_PROP, C_RC
#   pca, scaler, subject_to_id, answer_to_id
# Consumed by the propositional-adapter and engine-init blocks below.
# ══════════════════════════════════════════════════════════════════

# ── Token ID resolution (for R_base) ────────────────────────────
def resolve_ids(tok):
    ids = {}
    for ch in ['A','B','C','D']:
        for cand in [ch, f" {ch}", f"\n{ch}"]:
            ts = tok.encode(cand, add_special_tokens=False)
            if len(ts) == 1: ids[ch] = ts[0]; break
        else:
            ts = tok.encode(ch, add_special_tokens=False); ids[ch] = ts[-1]
    return ids

CHOICE_TOKEN_IDS = resolve_ids(tokenizer)
CHOICE_IDS_ARRAY = [CHOICE_TOKEN_IDS[c] for c in CHOICE_LABELS]
print(f"  Token IDs: {', '.join(f'{c}→{t}' for c,t in CHOICE_TOKEN_IDS.items())}")
print(f"  Using MAX_TOKEN_LEN={MAX_TOKEN_LEN} from Cell 3")

# ── Generic hidden-state extractor ───────────────────────────────
@torch.no_grad()
def extract_hidden(prompts, bs=16):
    """Extract last-token hidden state for each prompt. Returns (N, HIDDEN_DIM)."""
    zs = []
    for s in range(0, len(prompts), bs):
        b = prompts[s:s+bs]
        tokenizer.padding_side = 'left'
        enc = tokenizer(b, return_tensors='pt', padding=True,
                        truncation=True, max_length=MAX_TOKEN_LEN).to(DEVICE)
        out = model(**enc, output_hidden_states=True)
        zs.append(out.hidden_states[-1][:, -1, :].float().cpu().numpy())
    return np.concatenate(zs, axis=0)

@torch.no_grad()
def extract_base(prompts, bs=8):
    """Extract R_base_probs AND z_question from the base ABCD prompt."""
    zs, ps = [], []
    for s in range(0, len(prompts), bs):
        b = prompts[s:s+bs]
        tokenizer.padding_side = 'left'
        enc = tokenizer(b, return_tensors='pt', padding=True,
                        truncation=True, max_length=min(512, MAX_TOKEN_LEN * 2)).to(DEVICE)
        out = model(**enc, output_hidden_states=True)
        zs.append(out.hidden_states[-1][:, -1, :].float().cpu().numpy())
        lg = out.logits[:, -1, :][:, CHOICE_IDS_ARRAY]
        ps.append(F.softmax(lg.float(), dim=-1).cpu().numpy())
    return np.concatenate(zs, axis=0), np.concatenate(ps, axis=0)

# ── Run dual extraction ──────────────────────────────────────────
precomputed = {}

for pn in PHASE_NAMES:
    for sp in ['train', 'test']:
        rows = phase_data[pn][sp]
        n = len(rows)
        labels = np.array([r['label'] for r in rows], dtype=np.int32)

        # ── A. Base prompt → R_base_probs + z_question ──
        base_prompts = [r['base_prompt'] for r in rows]
        print(f"\n  {pn}/{sp} - base ({n})...", end="", flush=True)
        t0 = time.time()
        z_question_raw, R_base = extract_base(base_prompts)
        print(f" {time.time()-t0:.1f}s", end="")

        # ── B. Proposition prompts → z_choice_j ──
        # Format: "Question: {q}\nAnswer: {choice_text}"
        prop_prompts = []
        for r in rows:
            for j in range(N_CHOICES):
                prop_prompts.append(f"Question: {r['question']}\nAnswer: {r['choices'][j]}")

        print(f"  props ({len(prop_prompts)})...", end="", flush=True)
        t1 = time.time()
        z_props_raw = extract_hidden(prop_prompts)
        # Reshape: (N*4, HIDDEN_DIM) → (N, 4, HIDDEN_DIM)
        z_choices_raw = z_props_raw.reshape(n, N_CHOICES, HIDDEN_DIM)
        print(f" {time.time()-t1:.1f}s")

        # ── Store raw ──
        base_acc = (R_base.argmax(1) == labels).mean()
        precomputed[f"{pn}_{sp}"] = {
            'z_question_raw': z_question_raw,   # (N, 4096)
            'z_choices_raw': z_choices_raw,       # (N, 4, 4096)
            'R_base_probs': R_base,               # (N, 4)
            'labels': labels,                      # (N,)
        }
        print(f"    ✓ z_q={z_question_raw.shape}, z_ch={z_choices_raw.shape}, "
              f"base_acc={base_acc:.4f}")

# Free GPU
del model
if torch.cuda.is_available(): torch.cuda.empty_cache()
warnings.filterwarnings("default")
print(f"\n{'=' * 60}")
print(f"  ✓ All extracted (propositional), GPU freed")
print(f"{'=' * 60}")


# ══════════════════════════════════════════════════════════════════
# CELL 5: Sentence embeddings: mpnet proposition geometry
# ══════════════════════════════════════════════════════════════════


from sentence_transformers import SentenceTransformer

print("=" * 60)
print("  CELL 5: SENTENCE EMBEDDINGS (mpnet-base-v2)")
print("=" * 60)

sent_model = SentenceTransformer('all-mpnet-base-v2', device=str(DEVICE))
HIDDEN_DIM = 768
print(f"  ✓ Loaded: dim={HIDDEN_DIM}")

for pn in PHASE_NAMES:
    for sp in ['train', 'test']:
        k = f"{pn}_{sp}"
        rows = phase_data[pn][sp]

        # Question embeddings (agency space - uses full question text)
        q_texts = [r['question'] for r in rows]
        z_q = sent_model.encode(q_texts, batch_size=64,
                                show_progress_bar=False).astype(np.float32)

        # Proposition embeddings (value space - subject+choice only)
        # This collapses paraphrase distance to 0.
        prop_texts = []
        for r in rows:
            for j in range(N_CHOICES):
                prop_texts.append(f"{r['subject']} {r['choices'][j]}")
        z_props = sent_model.encode(prop_texts, batch_size=64,
                                    show_progress_bar=False).astype(np.float32)
        z_ch = z_props.reshape(len(rows), N_CHOICES, HIDDEN_DIM)

        precomputed[k]['z_question_raw'] = z_q
        precomputed[k]['z_choices_raw'] = z_ch

        print(f"  {k}: z_q={z_q.shape}, z_ch={z_ch.shape}")

del sent_model
if torch.cuda.is_available(): torch.cuda.empty_cache()
print("  ✓ Sentence embeddings extracted (mpnet-base-v2, subject+choice)")


# ══════════════════════════════════════════════════════════════════
# CELL 5b: REPRESENTATION + OPERATING GEOMETRY CALIBRATION — FINAL V6
# Clean 9B/9C-aligned replacement for old Cell 5b
# ══════════════════════════════════════════════════════════════════
#
# What this cell now does:
#   1. Builds the 80D proposition geometry:
#        mpnet raw → scaler + PCA 64D → + subject 8D → + answer 8D
#
#   2. Computes one runtime bandwidth only:
#        SIGMA_PROP = σ_operating
#
#   3. Treats the pairwise/sibling σ only as a bound:
#        SIGMA_BOUND_PAIRWISE
#
#   4. Treats dense-field aggregate pressure as the active operating bound:
#        SIGMA_BOUND_FIELD
#
#   5. Uses the 9B/9C-selected default:
#        ε_global = 5e-4
#        N_eff    = 10000
#
#   6. Preserves downstream notebook contract:
#        SIGMA_PROP, SIGMA_AGENCY, MERGE_PROP, C_RC
#        pca, scaler, subject_to_id, answer_to_id
#        FROZEN_HELDOUT, RETRACTION populations, geometry reports
#
# IMPORTANT:
#   This cell intentionally does NOT inherit old globals named
#   GEOMETRY_TOTAL_TAIL_BUDGET or GEOMETRY_EFFECTIVE_ACTIVE_DENSITY.
#   Those old globals caused Cell 5b V5 to silently keep ε=0.05.
#
#   To override this cell deliberately, set:
#        CELL6_EPS_GLOBAL_OVERRIDE = ...
#        CELL6_N_EFF_OVERRIDE = ...
#
# Operating law:
#
#   K(d, σ) = exp(-d² / 2σ²)
#
#   σ_operating = max σ such that:
#       K(d_pair, σ) <= ε_pair
#       N_eff · K(d_shell, σ) <= ε_global
#
#   Equivalent:
#
#       σ_operating = min(
#           d_pair  / sqrt(2 log(1 / ε_pair)),
#           d_shell / sqrt(2 log(N_eff / ε_global))
#       )
#
# Expected FI result with ε_global=5e-4:
#   σ_operating ≈ 7.2621
#   κ_operating ≈ 1.2672
#
# ══════════════════════════════════════════════════════════════════

import os, json, math, random, pickle, gc, time
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.spatial.distance import cdist

# ------------------------------------------------------------------
# 0. Representation configuration
# ------------------------------------------------------------------

Z_DIM = 64
SUBJECT_DIM = 8
ANSWER_DIM = 8
SUBJECT_SCALE = 25.0
ANSWER_SCALE = 25.0
Z_DIM_AUG = Z_DIM + SUBJECT_DIM + ANSWER_DIM  # 80D
P0 = PHASE_NAMES[0]

# ------------------------------------------------------------------
# 0b. Locked final geometry configuration from 9B/9C
# ------------------------------------------------------------------

PAIRWISE_BLEED_EPS = float(globals().get("CELL6_PAIRWISE_EPS_OVERRIDE", 1e-3))

# These two are intentionally isolated from old notebook globals.
CELL6_EPS_GLOBAL = float(globals().get("CELL6_EPS_GLOBAL_OVERRIDE", 5e-4))
CELL6_N_EFF = int(globals().get("CELL6_N_EFF_OVERRIDE", 10000))

# Publish canonical names for later cells/reports.
GEOMETRY_TOTAL_TAIL_BUDGET = float(CELL6_EPS_GLOBAL)
GEOMETRY_EFFECTIVE_ACTIVE_DENSITY = int(CELL6_N_EFF)

MERGE_TO_SIGMA_RATIO = float(globals().get("CELL6_MERGE_TO_SIGMA_RATIO_OVERRIDE", 1.5))

# Optional explicit sigma override for ablation only.
CELL6_SIGMA_OPERATING_OVERRIDE = globals().get("CELL6_SIGMA_OPERATING_OVERRIDE", None)
CELL6_SIGMA_AGENCY_OVERRIDE = globals().get("CELL6_SIGMA_AGENCY_OVERRIDE", None)
CELL6_MERGE_OVERRIDE = globals().get("CELL6_MERGE_OVERRIDE", None)

# Diagnostic tables only.
CELL6_SIGMA_MULTIPLIERS = list(globals().get(
    "CELL6_SIGMA_MULTIPLIERS_OVERRIDE",
    [0.50, 0.625, 0.70, 0.75, 0.80, 0.875, 1.00, 1.125, 1.25],
))

CELL6_EPSILON_CANDIDATES = list(globals().get(
    "CELL6_EPSILON_CANDIDATES_OVERRIDE",
    [5e-4, 1e-3, 2.5e-3, 5e-3, 1e-2, 2.5e-2, 5e-2],
))

GEOMETRY_JSON_PATH = os.path.join(OUT_DIR, "representation_geometry_calibration.json")
GEOMETRY_MD_PATH = os.path.join(OUT_DIR, "representation_geometry_calibration.md")

print("=" * 74)
print("  CELL 5b: REPRESENTATION + OPERATING GEOMETRY CALIBRATION — FINAL V6")
print("=" * 74)

print(f"""
Geometry policy:
  - one runtime σ only: SIGMA_PROP = σ_operating
  - pairwise locality is a bound / diagnostic, not a runtime σ
  - aggregate field pressure is the finite-field operating constraint
  - this cell is locked to the 9B/9C selected hygiene budget unless explicitly overridden

Locked defaults:
  ε_pair    = {PAIRWISE_BLEED_EPS:g}
  ε_global  = {GEOMETRY_TOTAL_TAIL_BUDGET:g}
  N_eff     = {GEOMETRY_EFFECTIVE_ACTIVE_DENSITY}
""")

# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------

def _kernel(d, sigma):
    d = np.asarray(d, dtype=np.float64)
    return np.exp(-(d ** 2) / (2.0 * float(sigma) ** 2))

def _sigma_for_bleed(distance, bleed):
    bleed = max(float(bleed), 1e-300)
    return float(distance) / np.sqrt(2.0 * np.log(1.0 / bleed))

def _sigma_for_aggregate(distance, n_eff, eps_global):
    n_eff = max(float(n_eff), 1.0)
    eps_global = max(float(eps_global), 1e-300)
    return float(distance) / np.sqrt(2.0 * np.log(n_eff / eps_global))

def _jsonify(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    raise TypeError(f"Object of type {type(obj)} not JSON serializable")

def _write_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=_jsonify, ensure_ascii=False)

def _markdown_table(rows, columns):
    if not rows:
        return ""
    def fmt(x):
        if x is None:
            return ""
        if isinstance(x, bool):
            return "✓" if x else "✗"
        if isinstance(x, float):
            if abs(x) >= 1000:
                return f"{x:.1f}"
            if abs(x) < 1e-4 and x != 0:
                return f"{x:.2e}"
            return f"{x:.4f}"
        return str(x)
    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join(["---"] * len(columns)) + " |"
    lines = [header, sep]
    for r in rows:
        lines.append("| " + " | ".join(fmt(r.get(c)) for c in columns) + " |")
    return "\n".join(lines)

def _percentiles(xs):
    xs = np.asarray(xs, dtype=np.float64)
    xs = xs[np.isfinite(xs)]
    if xs.size == 0:
        return {"min": None, "p10": None, "median": None, "p90": None, "mean": None, "max": None}
    return {
        "min": float(np.min(xs)),
        "p10": float(np.percentile(xs, 10)),
        "median": float(np.percentile(xs, 50)),
        "p90": float(np.percentile(xs, 90)),
        "mean": float(np.mean(xs)),
        "max": float(np.max(xs)),
    }

def _torch_empty_cache_if_available():
    if "torch" in globals():
        try:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass

# ------------------------------------------------------------------
# 1. PCA on proposition embeddings
# ------------------------------------------------------------------

z_fit_props = precomputed[f"{P0}_train"]["z_choices_raw"].reshape(-1, HIDDEN_DIM)
print(f"  Fitting scaler+PCA on {z_fit_props.shape[0]} propositions ({HIDDEN_DIM}D)")

scaler = StandardScaler().fit(z_fit_props)
z_scaled = scaler.transform(z_fit_props)

n_comp = min(Z_DIM, z_scaled.shape[1], z_scaled.shape[0])
pca = PCA(n_components=n_comp, random_state=SEED).fit(z_scaled)

eig = pca.explained_variance_
d_eff_pca = float((np.sum(eig) ** 2) / np.sum(eig ** 2))

print(f"  {HIDDEN_DIM}D → {n_comp}D")
print(f"  Variance explained: {np.sum(pca.explained_variance_ratio_):.4f}")
print(f"  d_eff (PCA only): {d_eff_pca:.1f}")

# ------------------------------------------------------------------
# 2. Subject feature map
# ------------------------------------------------------------------

all_subjects = set()
for pn in PHASE_NAMES:
    for sp in ["train", "test"]:
        for r in phase_data[pn][sp]:
            all_subjects.add(r["subject"])

subject_to_id = {s: i for i, s in enumerate(sorted(all_subjects))}
N_SUBJECTS = len(subject_to_id)
print(f"  Unique subjects: {N_SUBJECTS}")

def subject_features(subject):
    """Deterministic 8D feature vector per subject/entity."""
    sid = subject_to_id[subject]
    rng_sub = np.random.RandomState(sid + 54321)
    return rng_sub.randn(SUBJECT_DIM).astype(np.float32) * SUBJECT_SCALE / np.sqrt(SUBJECT_DIM)

_subj_list = list(subject_to_id.keys())
_sf1 = subject_features(_subj_list[0])
_sf2 = subject_features(_subj_list[0])
assert np.allclose(_sf1, _sf2), "Subject features must be deterministic"
_sf3 = subject_features(_subj_list[1])
_subj_dist = float(np.linalg.norm(_sf1 - _sf3))
print(f"  Subject feature distance (different subjects): {_subj_dist:.2f}")

# ------------------------------------------------------------------
# 3. Answer feature map
# ------------------------------------------------------------------

all_answers = set()
for pn in PHASE_NAMES:
    for sp in ["train", "test"]:
        for r in phase_data[pn][sp]:
            for c in r["choices"]:
                all_answers.add(c)

answer_to_id = {a: i for i, a in enumerate(sorted(all_answers))}
N_ANSWERS = len(answer_to_id)
print(f"  Unique answers: {N_ANSWERS}")

def answer_features(answer):
    """Deterministic 8D feature vector per unique answer/choice."""
    aid = answer_to_id[answer]
    rng_ans = np.random.RandomState(aid + 98765)
    return rng_ans.randn(ANSWER_DIM).astype(np.float32) * ANSWER_SCALE / np.sqrt(ANSWER_DIM)

_ans_list = list(answer_to_id.keys())
_af1 = answer_features(_ans_list[0])
_af2 = answer_features(_ans_list[1])
_ans_dist = float(np.linalg.norm(_af1 - _af2))
print(f"  Answer feature distance (different answers): {_ans_dist:.2f}")

# ------------------------------------------------------------------
# 4. PCA projection helper
# ------------------------------------------------------------------

def project_pca(z_raw):
    """Project raw hidden vectors to PCA space."""
    orig_shape = z_raw.shape
    flat = z_raw.reshape(-1, HIDDEN_DIM)
    proj = pca.transform(scaler.transform(flat)).astype(np.float32)
    return proj.reshape(orig_shape[:-1] + (n_comp,))

# ------------------------------------------------------------------
# 5. Project + augment all datasets
# ------------------------------------------------------------------

print("  Projecting and augmenting all datasets...")

for k in precomputed:
    d = precomputed[k]

    d["z_question"] = project_pca(d["z_question_raw"])

    z_ch_pca = project_pca(d["z_choices_raw"])  # (N, 4, 64)

    rows = None
    for pn in PHASE_NAMES:
        for sp in ["train", "test"]:
            if k == f"{pn}_{sp}":
                rows = phase_data[pn][sp]
                break
        if rows is not None:
            break

    if rows is None:
        continue

    N_items = len(rows)
    z_ch_aug = np.zeros((N_items, N_CHOICES, Z_DIM_AUG), dtype=np.float32)

    for i, r in enumerate(rows):
        sf = subject_features(r["subject"])
        for j in range(N_CHOICES):
            af = answer_features(r["choices"][j])
            z_ch_aug[i, j] = np.concatenate([z_ch_pca[i, j], sf, af])

    d["z_choices"] = z_ch_aug
    print(f"    {k}: z_q={d['z_question'].shape}, z_ch={d['z_choices'].shape}")

encoded_datasets = precomputed
encoded = precomputed
encoded_data = precomputed

# ------------------------------------------------------------------
# 6. Semantic separation check
# ------------------------------------------------------------------

print(f"\n  Semantic separation check (augmented {Z_DIM_AUG}D):")

semantic_rows = {}
all_wrong_dists = []

for pn in PHASE_NAMES:
    k = f"{pn}_train"
    if k not in precomputed:
        continue

    d = precomputed[k]
    zch = d["z_choices"]
    y = d["labels"]

    correct_dists, wrong_dists = [], []

    for i in range(min(200, len(y))):
        z_correct = zch[i, y[i]]
        for j in range(N_CHOICES):
            if j == y[i]:
                continue
            val = float(np.linalg.norm(z_correct - zch[i, j]))
            wrong_dists.append(val)
            all_wrong_dists.append(val)
        if i > 0:
            correct_dists.append(float(np.linalg.norm(zch[i, y[i]] - zch[i-1, y[i-1]])))

    semantic_rows[pn] = {
        "within_correct_wrong_mean": float(np.mean(wrong_dists)) if wrong_dists else None,
        "within_correct_wrong_stats": _percentiles(wrong_dists),
        "cross_correct_correct_mean": float(np.mean(correct_dists)) if correct_dists else None,
        "cross_correct_correct_stats": _percentiles(correct_dists),
    }

    print(
        f"    {pn}: within-Q correct↔wrong={np.mean(wrong_dists):.2f}, "
        f"cross-Q correct↔correct={np.mean(correct_dists):.2f}"
    )

# ------------------------------------------------------------------
# 7. Paraphrase distance
# ------------------------------------------------------------------

print(f"\n  ══ PARAPHRASE DISTANCE (augmented {Z_DIM_AUG}D) ══")

paraphrase_rows = {}
all_para_dists = []

for pn in PHASE_NAMES:
    k_tr = f"{pn}_train"
    k_te = f"{pn}_test"
    if k_tr not in precomputed or k_te not in precomputed:
        continue

    zch_tr = precomputed[k_tr]["z_choices"]
    zch_te = precomputed[k_te]["z_choices"]
    y_tr = precomputed[k_tr]["labels"]
    y_te = precomputed[k_te]["labels"]

    para_dists = []
    n_te = len(y_te)
    n_tr = len(y_tr)
    tmpl_ratio = max(1, n_tr // n_te) if n_te > 0 else 1

    for i in range(min(200, n_te)):
        tr_idx = i * tmpl_ratio
        if tr_idx >= n_tr:
            break
        d_val = float(np.linalg.norm(zch_tr[tr_idx, y_tr[tr_idx]] - zch_te[i, y_te[i]]))
        para_dists.append(d_val)
        all_para_dists.append(d_val)

    pm = float(np.mean(para_dists)) if para_dists else 0.0
    pmed = float(np.median(para_dists)) if para_dists else 0.0
    paraphrase_rows[pn] = {
        "mean": pm,
        "median": pmed,
        "stats": _percentiles(para_dists),
    }

    print(f"    {pn}: mean={pm:.2f}, median={pmed:.2f}")

# ------------------------------------------------------------------
# 8. Operating σ calibration
# ------------------------------------------------------------------

print(f"\n  σ calibration (augmented {Z_DIM_AUG}D)...")

cal_ch = precomputed[f"{P0}_train"]["z_choices"].reshape(-1, Z_DIM_AUG)[:2000]
cal_q = precomputed[f"{P0}_train"]["z_question"][:500]

rng_cal = np.random.RandomState(SEED)

# Random proposition distances.
i1 = rng_cal.randint(0, len(cal_ch), 5000)
i2 = rng_cal.randint(0, len(cal_ch), 5000)
ok = i1 != i2
d_prop = np.linalg.norm(cal_ch[i1[ok]] - cal_ch[i2[ok]], axis=1)
p10_prop = float(np.percentile(d_prop, 10))
p50_prop = float(np.median(d_prop))

# Within-question sibling distances: correct vs wrong choices.
sib_dists = []
zch_cal = precomputed[f"{P0}_train"]["z_choices"][:200]
y_cal = precomputed[f"{P0}_train"]["labels"][:200]

for i in range(len(y_cal)):
    z_c = zch_cal[i, y_cal[i]]
    for j in range(N_CHOICES):
        if j == y_cal[i]:
            continue
        sib_dists.append(float(np.linalg.norm(z_c - zch_cal[i, j])))

sib_med = float(np.median(sib_dists))

# Agency question-space distances.
i1q = rng_cal.randint(0, len(cal_q), 3000)
i2q = rng_cal.randint(0, len(cal_q), 3000)
okq = i1q != i2q
d_q = np.linalg.norm(cal_q[i1q[okq]] - cal_q[i2q[okq]], axis=1)
p10_q = float(np.percentile(d_q, 10))

# Effective dimensionality of augmented proposition space.
eig_aug = np.var(cal_ch, axis=0)
d_eff = float((np.sum(eig_aug) ** 2) / np.sum(eig_aug ** 2))
sqrt_d_eff = float(np.sqrt(d_eff))

# Pairwise locality bound.
# Diagnostic/upper bound only, not runtime σ.
d_pair = float(sib_med)
SIGMA_BOUND_PAIRWISE = float(_sigma_for_bleed(d_pair, PAIRWISE_BLEED_EPS))
KAPPA_BOUND_PAIRWISE = float(SIGMA_BOUND_PAIRWISE / sqrt_d_eff)

# Ratio for agency shrinkage.
SIGMA_AGENCY_BOUND_PAIRWISE = float(max(1.0, p10_q / np.sqrt(2.0 * np.log(100))))
AGENCY_TO_VALUE_RATIO = float(SIGMA_AGENCY_BOUND_PAIRWISE / max(SIGMA_BOUND_PAIRWISE, 1e-8))

# Conservative shell distance.
# The relevant safety shell is the closest high-density interference boundary.
d_shell = float(min(d_pair, p10_prop))

# Aggregate field-pressure bound.
SIGMA_BOUND_FIELD = float(_sigma_for_aggregate(
    d_shell,
    GEOMETRY_EFFECTIVE_ACTIVE_DENSITY,
    GEOMETRY_TOTAL_TAIL_BUDGET,
))
KAPPA_BOUND_FIELD = float(SIGMA_BOUND_FIELD / sqrt_d_eff)

# One runtime σ.
SIGMA_OPERATING_FORMULA = float(min(SIGMA_BOUND_PAIRWISE, SIGMA_BOUND_FIELD))

if CELL6_SIGMA_OPERATING_OVERRIDE is not None:
    SIGMA_OPERATING = float(CELL6_SIGMA_OPERATING_OVERRIDE)
    SIGMA_OPERATING_SOURCE = "explicit CELL6_SIGMA_OPERATING_OVERRIDE"
else:
    SIGMA_OPERATING = float(SIGMA_OPERATING_FORMULA)
    SIGMA_OPERATING_SOURCE = "constraint formula"

SIGMA_OPERATING_MULTIPLIER = float(SIGMA_OPERATING / max(SIGMA_BOUND_PAIRWISE, 1e-8))

MERGE_OPERATING = float(
    CELL6_MERGE_OVERRIDE
    if CELL6_MERGE_OVERRIDE is not None
    else SIGMA_OPERATING * MERGE_TO_SIGMA_RATIO
)

SIGMA_AGENCY_OPERATING = float(
    CELL6_SIGMA_AGENCY_OVERRIDE
    if CELL6_SIGMA_AGENCY_OVERRIDE is not None
    else SIGMA_OPERATING * AGENCY_TO_VALUE_RATIO
)

KAPPA_OPERATING = float(SIGMA_OPERATING / sqrt_d_eff)

active_constraint = "aggregate field" if SIGMA_BOUND_FIELD < SIGMA_BOUND_PAIRWISE else "pairwise locality"

# Runtime globals consumed downstream.
SIGMA_PROP = float(SIGMA_OPERATING)
MERGE_PROP = float(MERGE_OPERATING)
SIGMA_AGENCY = float(SIGMA_AGENCY_OPERATING)
kappa = float(KAPPA_OPERATING)

# Diagnostics.
K_pair_at_pairwise_bound = float(_kernel(d_pair, SIGMA_BOUND_PAIRWISE))
K_pair_operating = float(_kernel(d_pair, SIGMA_OPERATING))
K_shell_operating = float(_kernel(d_shell, SIGMA_OPERATING))
density_bound_operating = float(GEOMETRY_EFFECTIVE_ACTIVE_DENSITY * K_shell_operating)

K_p10_at_pairwise_bound = float(_kernel(p10_prop, SIGMA_BOUND_PAIRWISE))
K_p10_operating = float(_kernel(p10_prop, SIGMA_OPERATING))

print(f"  d_eff (augmented):         {d_eff:.4f}")
print(f"  Proposition P10={p10_prop:.3f}, median={p50_prop:.3f}")
print(f"  d_pair sibling median:     {d_pair:.3f}")
print(f"  d_shell safety boundary:   {d_shell:.3f}")
print(f"  Question P10={p10_q:.3f}")
print(f"  pairwise locality bound:   σ ≤ {SIGMA_BOUND_PAIRWISE:.4f}")
print(f"  aggregate field bound:     σ ≤ {SIGMA_BOUND_FIELD:.4f}")
print(f"  selected operating σ:      {SIGMA_OPERATING:.4f} ({SIGMA_OPERATING_SOURCE})")
print(f"  active constraint:          {active_constraint}")
print(f"  operating multiplier:       {SIGMA_OPERATING_MULTIPLIER:.4f}")
print(f"  operating agency σ:         {SIGMA_AGENCY_OPERATING:.4f}")
print(f"  operating merge:            {MERGE_OPERATING:.4f}")
print(f"  κ_pairwise_bound:           {KAPPA_BOUND_PAIRWISE:.4f}")
print(f"  κ_field_bound:              {KAPPA_BOUND_FIELD:.4f}")
print(f"  κ_operating:                {KAPPA_OPERATING:.4f}")
print(f"  ε_global:                   {GEOMETRY_TOTAL_TAIL_BUDGET:g}")
print(f"  N_eff:                      {GEOMETRY_EFFECTIVE_ACTIVE_DENSITY}")
print(f"  K(d_pair, σ_pair_bound):    {K_pair_at_pairwise_bound:.8f}")
print(f"  K(d_pair, σ_operating):     {K_pair_operating:.10f}")
print(f"  K(d_shell, σ_operating):    {K_shell_operating:.10f}")
print(f"  N_eff*K(d_shell):           {density_bound_operating:.8f}")
print(f"  K(P10_prop, σ_pair_bound):  {K_p10_at_pairwise_bound:.10f}")
print(f"  K(P10_prop, σ_operating):   {K_p10_operating:.10f}")

if abs(density_bound_operating - GEOMETRY_TOTAL_TAIL_BUDGET) > max(1e-8, GEOMETRY_TOTAL_TAIL_BUDGET * 1e-3):
    print("  NOTE: density bound is not exactly ε_global because another constraint or override is active.")

# ------------------------------------------------------------------
# 9. Kernel reach at paraphrase distance
# ------------------------------------------------------------------

print(f"\n  ══ KERNEL REACH AT PARAPHRASE DISTANCE ══")

kernel_reach_rows = {}
for pn in PHASE_NAMES:
    dist = paraphrase_rows.get(pn, {}).get("mean", 0.0) or 0.0
    kw = float(_kernel(dist, SIGMA_PROP))
    ratio = float(dist / SIGMA_PROP) if SIGMA_PROP > 0 else 0.0
    target_ok = "✓" if dist < 2.0 * SIGMA_PROP else "✗"
    kernel_reach_rows[pn] = {
        "dist": dist,
        "dist_over_sigma": ratio,
        "kernel": kw,
        "within_2sigma": target_ok,
    }
    print(f"    {pn}: dist={dist:.2f}, dist/σ={ratio:.2f}, kernel={kw:.6f}, <2σ={target_ok}")

# ------------------------------------------------------------------
# 10. Candidate geometry diagnostics
# ------------------------------------------------------------------

def _estimate_tail_mass(points, sigma, anchors=512, pool=4096):
    points = np.asarray(points, dtype=np.float32)
    n = points.shape[0]
    if n < 2:
        return {"mean": None, "p95": None, "max": None}

    rng = np.random.RandomState(SEED + int(round(float(sigma) * 1000)))
    anchor_idx = rng.choice(n, size=min(anchors, n), replace=False)
    pool_idx = rng.choice(n, size=min(pool, n), replace=False)

    A = points[anchor_idx]
    P = points[pool_idx]

    a_norm = np.sum(A ** 2, axis=1)[:, None]
    p_norm = np.sum(P ** 2, axis=1)[None, :]
    dist2 = np.maximum(a_norm + p_norm - 2.0 * (A @ P.T), 0.0)

    K = np.exp(-dist2 / (2.0 * float(sigma) ** 2))
    same = anchor_idx[:, None] == pool_idx[None, :]
    K[same] = 0.0

    sums = K.sum(axis=1)
    return {
        "mean": float(np.mean(sums)),
        "p95": float(np.percentile(sums, 95)),
        "max": float(np.max(sums)),
    }

cal_points = cal_ch.astype(np.float32)
positive_median = float(np.median(all_para_dists)) if all_para_dists else 0.0
candidate_rows = []

SAFE_DENSITY_TOL = max(1e-12, abs(GEOMETRY_TOTAL_TAIL_BUDGET) * 1e-9)

def _safe_density(density_tail):
    return bool(float(density_tail) <= float(GEOMETRY_TOTAL_TAIL_BUDGET) + SAFE_DENSITY_TOL)

# Sigma multipliers relative to pairwise bound.
for mult in CELL6_SIGMA_MULTIPLIERS:
    sig = float(SIGMA_BOUND_PAIRWISE * mult)
    k_pair = float(_kernel(d_pair, sig))
    k_shell = float(_kernel(d_shell, sig))
    density_tail = float(GEOMETRY_EFFECTIVE_ACTIVE_DENSITY * k_shell)
    kpos = float(_kernel(positive_median, sig))
    tail = _estimate_tail_mass(cal_points, sig)

    candidate_rows.append({
        "kind": "sigma_multiplier",
        "label": f"{mult:.3f}x_pair_bound",
        "multiplier": float(mult),
        "epsilon_global": None,
        "sigma": sig,
        "kappa": float(sig / sqrt_d_eff),
        "merge": float(sig * MERGE_TO_SIGMA_RATIO),
        "single_pair_bleed": k_pair,
        "single_shell_bleed": k_shell,
        "density_tail_bound": density_tail,
        "tail_mean_sample": tail["mean"],
        "tail_p95_sample": tail["p95"],
        "positive_kernel": kpos,
        "safe_density": _safe_density(density_tail),
        "formula_selected": False,
    })

# Epsilon-derived candidates.
for eps in CELL6_EPSILON_CANDIDATES:
    sig = float(min(
        SIGMA_BOUND_PAIRWISE,
        _sigma_for_aggregate(d_shell, GEOMETRY_EFFECTIVE_ACTIVE_DENSITY, eps),
    ))
    k_pair = float(_kernel(d_pair, sig))
    k_shell = float(_kernel(d_shell, sig))
    density_tail = float(GEOMETRY_EFFECTIVE_ACTIVE_DENSITY * k_shell)
    kpos = float(_kernel(positive_median, sig))
    tail = _estimate_tail_mass(cal_points, sig)

    candidate_rows.append({
        "kind": "epsilon_budget",
        "label": f"eps_{eps:g}",
        "multiplier": float(sig / max(SIGMA_BOUND_PAIRWISE, 1e-8)),
        "epsilon_global": float(eps),
        "sigma": sig,
        "kappa": float(sig / sqrt_d_eff),
        "merge": float(sig * MERGE_TO_SIGMA_RATIO),
        "single_pair_bleed": k_pair,
        "single_shell_bleed": k_shell,
        "density_tail_bound": density_tail,
        "tail_mean_sample": tail["mean"],
        "tail_p95_sample": tail["p95"],
        "positive_kernel": kpos,
        "safe_density": bool(density_tail <= GEOMETRY_TOTAL_TAIL_BUDGET),
        "formula_selected": False,
    })

# Mark exact selected formula row when possible.
if candidate_rows:
    exact_matches = [
        idx for idx, r in enumerate(candidate_rows)
        if r["kind"] == "epsilon_budget"
        and r["epsilon_global"] is not None
        and abs(float(r["epsilon_global"]) - GEOMETRY_TOTAL_TAIL_BUDGET) < 1e-15
    ]
    if exact_matches:
        candidate_rows[exact_matches[0]]["formula_selected"] = True
    else:
        nearest = int(np.argmin([abs(r["sigma"] - SIGMA_OPERATING) for r in candidate_rows]))
        candidate_rows[nearest]["formula_selected"] = True


print(f"\n  ══ CANDIDATE OPERATING GEOMETRIES ══")
print("  Diagnostic only. Runtime uses selected operating σ above.")

for r in candidate_rows:
    if r["kind"] == "epsilon_budget":
        prefix = f"ε={r['epsilon_global']:<9g}"
    else:
        prefix = f"σx={r['multiplier']:<9.3f}"

    mark = " ← selected" if r["formula_selected"] else ""
    print(
        f"    {prefix} "
        f"σ={r['sigma']:<8.4f} "
        f"κ={r['kappa']:<7.4f} "
        f"Kshell={r['single_shell_bleed']:.2e} "
        f"N*K={r['density_tail_bound']:.6f} "
        f"tail95={r['tail_p95_sample']:.4f} "
        f"safe={'✓' if r['safe_density'] else '✗'}"
        f"{mark}"
    )

# ------------------------------------------------------------------
# 11. Downstream config / compatibility globals
# ------------------------------------------------------------------

C_RC = {
    "SIGMA": float(SIGMA_PROP),
    "MERGE_THRESHOLD": float(MERGE_PROP),
    "V_MAX": 5.0,
    "DR_CAPACITY": 20000,
}

# Preferred explicit geometry globals.
IBF_SIGMA_BOUND_PAIRWISE = float(SIGMA_BOUND_PAIRWISE)
IBF_KAPPA_BOUND_PAIRWISE = float(KAPPA_BOUND_PAIRWISE)
IBF_SIGMA_BOUND_FIELD = float(SIGMA_BOUND_FIELD)
IBF_KAPPA_BOUND_FIELD = float(KAPPA_BOUND_FIELD)

IBF_OPERATING_SIGMA = float(SIGMA_PROP)
IBF_OPERATING_SIGMA_MULTIPLIER = float(SIGMA_OPERATING_MULTIPLIER)
IBF_OPERATING_SIGMA_AGENCY = float(SIGMA_AGENCY)
IBF_OPERATING_MERGE_THRESHOLD = float(MERGE_PROP)
IBF_OPERATING_KAPPA = float(kappa)
IBF_OPERATING_EPS_GLOBAL = float(GEOMETRY_TOTAL_TAIL_BUDGET)
IBF_OPERATING_N_EFF = int(GEOMETRY_EFFECTIVE_ACTIVE_DENSITY)

IBF_D_EFF = float(d_eff)
IBF_SQRT_D_EFF = float(sqrt_d_eff)
IBF_D_PAIR = float(d_pair)
IBF_D_SHELL = float(d_shell)

# Legacy aliases only. Do not use conceptually downstream.
IBF_BASE_SIGMA = float(SIGMA_BOUND_PAIRWISE)
IBF_BASE_SIGMA_AGENCY = float(SIGMA_AGENCY_BOUND_PAIRWISE)
IBF_BASE_MERGE_THRESHOLD = float(SIGMA_BOUND_PAIRWISE * MERGE_TO_SIGMA_RATIO)
IBF_BASE_KAPPA = float(KAPPA_BOUND_PAIRWISE)

# More compatibility names for later patched cells.
sigma_value = float(SIGMA_PROP)
sigma_agency = float(SIGMA_AGENCY)
merge_threshold = float(MERGE_PROP)
LOCKED_SIGMA = float(SIGMA_PROP)
LOCKED_MERGE = float(MERGE_PROP)

# ------------------------------------------------------------------
# 12. P6b - near-neighbor + distant control matching
# ------------------------------------------------------------------

print(f"\n  ══ P6b: RETRACTION CONTROL MATCHING ══")

from sentence_transformers import SentenceTransformer

print("  Loading mpnet for proposition matching...")
_sent_model_p6b = SentenceTransformer("all-mpnet-base-v2", device=str(DEVICE))

def compute_augmented_proposition_batch(subjects, answers):
    """Batch compute 80D augmented proposition vectors."""
    prop_texts = [f"{s} {a}" for s, a in zip(subjects, answers)]
    z_raw = _sent_model_p6b.encode(
        prop_texts,
        batch_size=64,
        show_progress_bar=False,
    ).astype(np.float32)

    z_pca_all = pca.transform(scaler.transform(z_raw)).astype(np.float32)

    results = []
    for i, (subj, ans) in enumerate(zip(subjects, answers)):
        sf = subject_features(subj)
        af = answer_features(ans)
        results.append(np.concatenate([z_pca_all[i], sf, af]))

    return np.array(results, dtype=np.float32)

pa_names = [e["name"] for e in phase_a_emps]
pa_manager_answers = [e["manager"] for e in phase_a_emps]

pa_mgr_props = compute_augmented_proposition_batch(pa_names, pa_manager_answers)
print(f"  Manager propositions computed: {pa_mgr_props.shape}")

target_indices = [i for i, n in enumerate(pa_names) if n in RETRACTION_TARGETS_SET]
nontarget_indices = [i for i, n in enumerate(pa_names) if n not in RETRACTION_TARGETS_SET]

assert len(target_indices) == 50, f"Expected 50 target indices, got {len(target_indices)}"
assert len(nontarget_indices) == 150, f"Expected 150 non-target indices, got {len(nontarget_indices)}"

target_mgr_props = pa_mgr_props[target_indices]
nontarget_mgr_props = pa_mgr_props[nontarget_indices]
nontarget_names = [pa_names[i] for i in nontarget_indices]

mgr_dists = cdist(target_mgr_props, nontarget_mgr_props, metric="euclidean")

claimed = set()
near_neighbor_map = {}
near_neighbor_distances = {}

target_min_dists = [(t_idx, float(np.min(mgr_dists[t_idx]))) for t_idx in range(50)]
target_min_dists.sort(key=lambda x: x[1])

for t_idx, _ in target_min_dists:
    t_name = pa_names[target_indices[t_idx]]
    sorted_nt = np.argsort(mgr_dists[t_idx])
    for nt_rank_idx in sorted_nt:
        if nt_rank_idx not in claimed:
            claimed.add(nt_rank_idx)
            nt_name = nontarget_names[nt_rank_idx]
            near_neighbor_map[t_name] = nt_name
            near_neighbor_distances[t_name] = float(mgr_dists[t_idx, nt_rank_idx])
            break

NEAR_NEIGHBOR_CONTROLS = sorted(near_neighbor_map.values())
NEAR_NEIGHBOR_CONTROLS_SET = set(NEAR_NEIGHBOR_CONTROLS)

assert len(NEAR_NEIGHBOR_CONTROLS) == 50, \
    f"Expected 50 near-neighbor controls, got {len(NEAR_NEIGHBOR_CONTROLS)}"

nn_dists = list(near_neighbor_distances.values())

print(f"\n  Near-neighbor matching (manager category, greedy):")
print(f"    50 targets → 50 unique near-neighbors")
print(
    f"    Distance stats: min={np.min(nn_dists):.2f}, "
    f"median={np.median(nn_dists):.2f}, max={np.max(nn_dists):.2f}"
)
print(
    f"    Distance in operating σ units: min={np.min(nn_dists)/SIGMA_PROP:.2f}σ, "
    f"median={np.median(nn_dists)/SIGMA_PROP:.2f}σ, "
    f"max={np.max(nn_dists)/SIGMA_PROP:.2f}σ"
)

# Distant controls.
remaining_indices = [i for i in nontarget_indices if pa_names[i] not in NEAR_NEIGHBOR_CONTROLS_SET]
remaining_names = [pa_names[i] for i in remaining_indices]
print(f"\n  Distant control pool: {len(remaining_names)} candidates")

pa_team_answers = [e["team"] for e in phase_a_emps]
pa_project_answers = [e["project"] for e in phase_a_emps]

target_team_props = compute_augmented_proposition_batch(
    [pa_names[i] for i in target_indices],
    [pa_team_answers[i] for i in target_indices],
)
target_prj_props = compute_augmented_proposition_batch(
    [pa_names[i] for i in target_indices],
    [pa_project_answers[i] for i in target_indices],
)

rem_mgr_props = compute_augmented_proposition_batch(
    remaining_names,
    [pa_manager_answers[remaining_indices[j]] for j in range(len(remaining_indices))],
)
rem_team_props = compute_augmented_proposition_batch(
    remaining_names,
    [pa_team_answers[remaining_indices[j]] for j in range(len(remaining_indices))],
)
rem_prj_props = compute_augmented_proposition_batch(
    remaining_names,
    [pa_project_answers[remaining_indices[j]] for j in range(len(remaining_indices))],
)

n_rem = len(remaining_names)
min_dist_to_any_target = np.full(n_rem, np.inf)

for cat_target, cat_remain in [
    (target_mgr_props, rem_mgr_props),
    (target_team_props, rem_team_props),
    (target_prj_props, rem_prj_props),
]:
    d_mat = cdist(cat_remain, cat_target, metric="euclidean")
    d_min_per_rem = np.min(d_mat, axis=1)
    min_dist_to_any_target = np.minimum(min_dist_to_any_target, d_min_per_rem)

DISTANT_THRESHOLD_USED = 3.0 * SIGMA_PROP
distant_mask = min_dist_to_any_target > DISTANT_THRESHOLD_USED
n_distant = int(np.sum(distant_mask))

if n_distant < 50:
    n_at_3sigma = int(np.sum(min_dist_to_any_target > 3.0 * SIGMA_PROP))
    DISTANT_THRESHOLD_USED = 2.5 * SIGMA_PROP
    distant_mask = min_dist_to_any_target > DISTANT_THRESHOLD_USED
    n_distant = int(np.sum(distant_mask))
    print(f"  ⚠ 3σ threshold yielded {n_at_3sigma} candidates, relaxed to 2.5σ → {n_distant} candidates")

if n_distant < 50:
    DISTANT_THRESHOLD_USED = float(np.sort(min_dist_to_any_target)[-50])
    distant_mask = min_dist_to_any_target >= DISTANT_THRESHOLD_USED
    n_distant = int(np.sum(distant_mask))
    print(
        f"  ⚠ 2.5σ still insufficient, forcing top-50 by distance "
        f"(threshold={DISTANT_THRESHOLD_USED:.2f} = {DISTANT_THRESHOLD_USED/SIGMA_PROP:.2f}σ)"
    )

distant_candidates = [remaining_names[i] for i in range(n_rem) if distant_mask[i]]
distant_distances = [float(min_dist_to_any_target[i]) for i in range(n_rem) if distant_mask[i]]

if len(distant_candidates) > 50:
    _dist_rng = random.Random(RETRACTION_TARGET_SEED + 1)
    _dist_sample = _dist_rng.sample(range(len(distant_candidates)), 50)
    DISTANT_CONTROLS = sorted([distant_candidates[i] for i in _dist_sample])
    distant_sampled_distances = [distant_distances[i] for i in _dist_sample]
else:
    DISTANT_CONTROLS = sorted(distant_candidates[:50])
    distant_sampled_distances = distant_distances[:50]

DISTANT_CONTROLS_SET = set(DISTANT_CONTROLS)

print(f"\n  Distant controls:")
print(f"    Threshold: {DISTANT_THRESHOLD_USED:.2f} ({DISTANT_THRESHOLD_USED/SIGMA_PROP:.2f}σ)")
print(f"    Candidates passing threshold: {len(distant_candidates)}")
print(f"    Selected: {len(DISTANT_CONTROLS)}")

if distant_sampled_distances:
    print(
        f"    Distance stats: min={np.min(distant_sampled_distances):.2f} "
        f"({np.min(distant_sampled_distances)/SIGMA_PROP:.2f}σ), "
        f"median={np.median(distant_sampled_distances):.2f} "
        f"({np.median(distant_sampled_distances)/SIGMA_PROP:.2f}σ)"
    )

assert len(RETRACTION_TARGETS_SET & NEAR_NEIGHBOR_CONTROLS_SET) == 0, \
    "Overlap between targets and near-neighbor controls"
assert len(RETRACTION_TARGETS_SET & DISTANT_CONTROLS_SET) == 0, \
    "Overlap between targets and distant controls"
assert len(NEAR_NEIGHBOR_CONTROLS_SET & DISTANT_CONTROLS_SET) == 0, \
    "Overlap between near-neighbor and distant controls"

print(
    f"  ✓ No overlap: {len(RETRACTION_TARGETS)} targets, "
    f"{len(NEAR_NEIGHBOR_CONTROLS)} near-neighbors, "
    f"{len(DISTANT_CONTROLS)} distant"
)

del _sent_model_p6b
_torch_empty_cache_if_available()

# ------------------------------------------------------------------
# 13. P3 - Frozen held-out set
# ------------------------------------------------------------------

print(f"\n  ══ P3: FROZEN HELD-OUT SET ══")

frozen_rng = random.Random(FROZEN_HELDOUT_SEED)
FROZEN_HELDOUT = []
a_test_items = phase_data["A_Onboarding"]["test"]

for cat in ["team", "manager", "project", "mentor", "location"]:
    cat_items = [t for t in a_test_items if t["category"] == cat]
    assert len(cat_items) >= 20, \
        f"Category {cat} has only {len(cat_items)} test items, need 20"
    sampled = frozen_rng.sample(cat_items, 20)
    FROZEN_HELDOUT.extend(sampled)

assert len(FROZEN_HELDOUT) == 100, \
    f"Expected 100 frozen items, got {len(FROZEN_HELDOUT)}"

_fh_cats = {}
for item in FROZEN_HELDOUT:
    _fh_cats[item["category"]] = _fh_cats.get(item["category"], 0) + 1

print(f"  Frozen held-out set: {len(FROZEN_HELDOUT)} items (seed {FROZEN_HELDOUT_SEED})")
for cat, n in sorted(_fh_cats.items()):
    print(f"    {cat}: {n}")

# ------------------------------------------------------------------
# 14. Save reports/artifacts
# ------------------------------------------------------------------

retraction_populations = {
    "target_subjects": RETRACTION_TARGETS,
    "near_neighbor_subjects": NEAR_NEIGHBOR_CONTROLS,
    "near_neighbor_map": near_neighbor_map,
    "near_neighbor_distances": near_neighbor_distances,
    "distant_subjects": DISTANT_CONTROLS,
    "distant_threshold": float(DISTANT_THRESHOLD_USED),
    "distant_threshold_sigma_units": float(DISTANT_THRESHOLD_USED / SIGMA_PROP),
    "sigma_operating": float(SIGMA_PROP),
    "sigma_bound_pairwise": float(SIGMA_BOUND_PAIRWISE),
    "sigma_bound_field": float(SIGMA_BOUND_FIELD),
    "sigma_operating_multiplier": float(SIGMA_OPERATING_MULTIPLIER),
    "epsilon_global": float(GEOMETRY_TOTAL_TAIL_BUDGET),
    "n_eff": int(GEOMETRY_EFFECTIVE_ACTIVE_DENSITY),
}

_pop_path = os.path.join(OUT_DIR, "retraction_populations.pkl")
with open(_pop_path, "wb") as f:
    pickle.dump(retraction_populations, f)
print(f"\n  Saved: {_pop_path}")

_fh_path = os.path.join(OUT_DIR, "frozen_heldout.pkl")
with open(_fh_path, "wb") as f:
    pickle.dump(FROZEN_HELDOUT, f)
print(f"  Saved: {_fh_path}")

geometry_artifact = {
    "schema_version": "cell6_geometry_operating_sigma.v6",
    "status": "9B/9C-aligned final geometry cell",
    "conceptual_policy": {
        "runtime_sigma": "SIGMA_PROP is the selected operating sigma",
        "pairwise_bound": "diagnostic upper bound only; not a runtime sigma",
        "field_bound": "aggregate field-pressure upper bound",
        "formula": "sigma_operating = min(sigma_pairwise_bound, sigma_field_bound)",
        "epsilon_global_source": "CELL6_EPS_GLOBAL_OVERRIDE or locked default 5e-4",
    },
    "representation": {
        "hidden_dim": int(HIDDEN_DIM),
        "z_dim": int(Z_DIM),
        "subject_dim": int(SUBJECT_DIM),
        "answer_dim": int(ANSWER_DIM),
        "z_dim_aug": int(Z_DIM_AUG),
        "variance_explained": float(np.sum(pca.explained_variance_ratio_)),
        "d_eff_pca": float(d_eff_pca),
        "d_eff_aug": float(d_eff),
        "sqrt_d_eff": float(sqrt_d_eff),
    },
    "distances": {
        "d_pair": float(d_pair),
        "d_shell": float(d_shell),
        "p10_prop": float(p10_prop),
        "p50_prop": float(p50_prop),
        "p10_question": float(p10_q),
        "sibling_distance_stats": _percentiles(sib_dists),
        "random_prop_distance_stats": _percentiles(d_prop),
        "question_distance_stats": _percentiles(d_q),
    },
    "bounds": {
        "pairwise": {
            "sigma_bound": float(SIGMA_BOUND_PAIRWISE),
            "kappa_bound": float(KAPPA_BOUND_PAIRWISE),
            "epsilon_pair": float(PAIRWISE_BLEED_EPS),
            "K_at_d_pair": float(K_pair_at_pairwise_bound),
        },
        "aggregate_field": {
            "sigma_bound": float(SIGMA_BOUND_FIELD),
            "kappa_bound": float(KAPPA_BOUND_FIELD),
            "epsilon_global": float(GEOMETRY_TOTAL_TAIL_BUDGET),
            "n_eff": int(GEOMETRY_EFFECTIVE_ACTIVE_DENSITY),
            "K_at_d_shell": float(_kernel(d_shell, SIGMA_BOUND_FIELD)),
            "density_bound": float(GEOMETRY_EFFECTIVE_ACTIVE_DENSITY * _kernel(d_shell, SIGMA_BOUND_FIELD)),
        },
    },
    "operating_geometry": {
        "sigma": float(SIGMA_PROP),
        "sigma_agency": float(SIGMA_AGENCY),
        "merge_threshold": float(MERGE_PROP),
        "kappa": float(kappa),
        "source": SIGMA_OPERATING_SOURCE,
        "active_constraint": active_constraint,
        "multiplier_vs_pairwise_bound": float(SIGMA_OPERATING_MULTIPLIER),
        "epsilon_global": float(GEOMETRY_TOTAL_TAIL_BUDGET),
        "n_eff": int(GEOMETRY_EFFECTIVE_ACTIVE_DENSITY),
        "K_d_pair": float(K_pair_operating),
        "K_d_shell": float(K_shell_operating),
        "N_eff_K_d_shell": float(density_bound_operating),
        "K_p10_prop": float(K_p10_operating),
    },
    "semantic_rows": semantic_rows,
    "paraphrase_rows": paraphrase_rows,
    "kernel_reach_rows": kernel_reach_rows,
    "candidate_rows": candidate_rows,
    "retraction_populations_path": _pop_path,
    "frozen_heldout_path": _fh_path,
}

_write_json(GEOMETRY_JSON_PATH, geometry_artifact)

md_rows = []
for r in candidate_rows:
    md_rows.append({
        "kind": r["kind"],
        "label": r["label"],
        "ε": r["epsilon_global"],
        "σ": r["sigma"],
        "κ": r["kappa"],
        "K_shell": r["single_shell_bleed"],
        "N_eff*K": r["density_tail_bound"],
        "tail_p95": r["tail_p95_sample"],
        "K_positive": r["positive_kernel"],
        "safe": r["safe_density"],
        "selected": r["formula_selected"],
    })

with open(GEOMETRY_MD_PATH, "w", encoding="utf-8") as f:
    f.write("# Cell 5b — representation + operating geometry V6\n\n")
    f.write("This cell derives the IBF correction-field operating bandwidth from active geometric constraints.\n\n")
    f.write("Runtime uses only one sigma: `σ_operating`.\n\n")

    f.write("## Core law\n\n")
    f.write("```text\n")
    f.write("K(d, σ) = exp(-d² / 2σ²)\n\n")
    f.write("σ_operating = max σ such that:\n")
    f.write("  K(d_pair, σ) <= ε_pair\n")
    f.write("  N_eff · K(d_shell, σ) <= ε_global\n")
    f.write("```\n\n")

    f.write("## Summary\n\n")
    f.write(f"- d_eff: `{d_eff:.6f}`\n")
    f.write(f"- d_pair: `{d_pair:.6f}`\n")
    f.write(f"- d_shell: `{d_shell:.6f}`\n")
    f.write(f"- pairwise locality bound: `σ <= {SIGMA_BOUND_PAIRWISE:.6f}`\n")
    f.write(f"- aggregate field bound: `σ <= {SIGMA_BOUND_FIELD:.6f}`\n")
    f.write(f"- selected operating σ: `{SIGMA_PROP:.6f}`\n")
    f.write(f"- selected operating κ: `{kappa:.6f}`\n")
    f.write(f"- active constraint: `{active_constraint}`\n")
    f.write(f"- ε_global: `{GEOMETRY_TOTAL_TAIL_BUDGET:g}`\n")
    f.write(f"- N_eff: `{GEOMETRY_EFFECTIVE_ACTIVE_DENSITY}`\n")
    f.write(f"- operating merge: `{MERGE_PROP:.6f}`\n")
    f.write(f"- operating agency σ: `{SIGMA_AGENCY:.6f}`\n")
    f.write(f"- density bound `N_eff*K(d_shell)`: `{density_bound_operating:.8f}`\n\n")

    f.write("## Candidate geometry table\n\n")
    f.write(_markdown_table(
        md_rows,
        ["kind", "label", "ε", "σ", "κ", "K_shell", "N_eff*K", "tail_p95", "K_positive", "safe", "selected"],
    ))
    f.write("\n")

print(f"\n  Saved: {GEOMETRY_JSON_PATH}")
print(f"  Saved: {GEOMETRY_MD_PATH}")

_torch_empty_cache_if_available()
gc.collect()

print(f"\n{'=' * 74}")
print(f"  ✓ PCA: {HIDDEN_DIM}D → {Z_DIM}D + {SUBJECT_DIM}D + {ANSWER_DIM}D = {Z_DIM_AUG}D")
print(f"  ✓ pairwise bound σ ≤ {SIGMA_BOUND_PAIRWISE:.4f}")
print(f"  ✓ field bound σ ≤ {SIGMA_BOUND_FIELD:.4f}")
print(f"  ✓ selected operating σ = {SIGMA_PROP:.4f}")
print(f"  ✓ operating κ = {kappa:.4f}")
print(f"  ✓ ε_global = {GEOMETRY_TOTAL_TAIL_BUDGET:g}, N_eff = {GEOMETRY_EFFECTIVE_ACTIVE_DENSITY}")
print(f"  ✓ σ_agency = {SIGMA_AGENCY:.4f}, merge = {MERGE_PROP:.4f}")
print(f"  ✓ density bound N_eff*K(d_shell) = {density_bound_operating:.8f}")
print(f"  ✓ P6b: 50 near-neighbor + {len(DISTANT_CONTROLS)} distant controls matched")
print(f"  ✓ P3: 100 frozen held-out items (20/category)")
print(f"  ✓ Populations saved to {OUT_DIR}/")
print(f"  ✓ Ready for Cell 6 (adapter)")
print(f"{'=' * 74}")

# ──────────────────────────────────────────────────────────────────
# Propositional adapter + FAISS (preserves v2's working integration)
# ──────────────────────────────────────────────────────────────────

# ----- Propositional adapter (binds engine to 80-D value, 64-D agency) -----
_ADAPTER_R_FIELD_VALUE = 0.5


def RC_encode(board):
    """Agency space: 64-D question vector."""
    return board


def RC_encode_move(board_before, board_after, move):
    """Value space: 80-D proposition vector."""
    return board_after


def RC_R_field(board):
    return _ADAPTER_R_FIELD_VALUE


def apply_move(b, m):
    return b


# ----- Build the engine at the operating geometry -----
p = IBFParams.from_calibration(C_RC)
p.sigma = SIGMA_PROP
p.merge_threshold = MERGE_PROP
p.sigma_agency = SIGMA_AGENCY
p.v_max = 5.0; p.w_max = 5.0
p.k_0 = 5.0; p.k_min = 1.0
p.eta = 0.5; p.eta_cryst = 0.015
p.eta_k = 0.05; p.eta_k_cryst = 0.005   # Reading C parallel learning rate
p.mu_base = 0.06; p.mu_crystallized = 0.001
p.crystallization_threshold = 15
p.convergence_threshold = 0.025
p.n_agency_min = 20                      # Reading C history-sufficiency gate
p.n_cross_min = 8
p.reversal_threshold = -0.0125
p.alpha_shrink = 10.0
p.sigma_floor = SIGMA_PROP * 0.25
p.min_samples_shrink = 50
p.capacity = 20000

ibf = IBFEngine(params=p)

# ----- FAISS acceleration wrapper -----
try:
    import faiss
except ImportError:
    faiss = None
    print("  ⚠ faiss not available; engine will use exact delta_R only")


class FAISSAccelerator:
    """Wraps IBFEngine with FAISS-accelerated kernel lookups.

    Storage-only wrapper — exact kernels are computed for the centers within
    `search_radius_sigma · σ` of the query. Centers beyond 3σ have kernel
    weight < activation_threshold, so the skip is exact, not approximate.
    """

    def __init__(self, engine, search_radius_sigma=3.0, max_neighbors=200):
        self.engine = engine
        self.search_radius_sigma = search_radius_sigma
        self.max_neighbors = max_neighbors
        self._value_index = None; self._agency_index = None
        self._value_positions = None; self._agency_positions = None

    def _build_index(self, positions, d):
        if len(positions) == 0:
            return None
        idx = faiss.IndexFlatL2(d)
        idx.add(positions.astype(np.float32))
        return idx

    def rebuild_index(self):
        if faiss is None:
            return
        if self.engine.value_centers:
            self._value_positions = np.array(
                [c.z for c in self.engine.value_centers], dtype=np.float32)
            d = self._value_positions.shape[1]
            self._value_index = self._build_index(self._value_positions, d)
        else:
            self._value_index = None
        if self.engine.agency_centers:
            self._agency_positions = np.array(
                [c.z for c in self.engine.agency_centers], dtype=np.float32)
            d = self._agency_positions.shape[1]
            self._agency_index = self._build_index(self._agency_positions, d)
        else:
            self._agency_index = None


faiss_acc = FAISSAccelerator(ibf) if faiss is not None else None

# Smoke-test the adapter end-to-end on one item.
P0 = PHASE_NAMES[0]
_zq = precomputed[f"{P0}_train"]["z_question"][0]
_zch = precomputed[f"{P0}_train"]["z_choices"][0, 0]
_rp = float(precomputed[f"{P0}_train"]["R_base_probs"][0, 0])
_ADAPTER_R_FIELD_VALUE = _rp
ibf.set_context(0)
r = ibf.compute_D_and_update(
    board_before=_zq, board_after_own_move=_zch,
    board_after_opponent=None, move_chosen=0, R_imposed_override=0.0)
print(f"  adapter smoke: D={r['D']:.4f}, val={len(ibf.value_centers)}, agn={len(ibf.agency_centers)}")
ibf = IBFEngine(params=p)
if faiss_acc:
    faiss_acc.engine = ibf
print(f"  ✓ Adapter + FAISS ready; engine reset for C1 training")


# Part II — The Eight Claims

The eight claim sections that follow are each a self-contained piece of evidence for one
property of the substrate. They build a four-layer dependency stack: Existence (C1) is the
foundation; Properties (C2-C4) describe what kind of object the substrate is; Operations
(C5-C6) describe what can be done with it; Composition (C7-C8) describes how it scales
into structured knowledge and discovery-driven extension.

---

## § C1 — Local durable alignment without weight editing

**Claim.** A frozen LLM can be locally aligned to durable corrections without modifying
its weights. Crystallised modifications survive dormant epochs and yield measurable
behavioural retention.

**Layer.** 1 (Existence).

**Foundational anchor.** Foundational Claim 1 (Memory) — *"If a visited region reaches
local thermodynamic equilibrium under Postulate 1, the induced modification undergoes a
stability transition (μ → 0) and persists as a long-lived structure in the effective
coherence landscape."* This card is the LLM-substrate instantiation of buffer-free
retention.

**Presupposes.** Nothing (foundational existence claim).

**Adds.** The substrate exists as a distinct architectural object.

**Enables.** Every subsequent claim.

**Falsifier (mirroring foundational discipline).** Crystallised modifications decay during
dormant epochs despite the stability transition, OR they persist structurally but yield
no measurable behavioural retention (avg lin indistinguishable from base model alone).

**Headline result to reproduce (within ±0.01 absolute):**
- avg lin = 0.954 (vs base 0.216, +0.738 absolute)
- 4 / 4 phases converged
- Engine end-state: 6382 value centers (all crystallised), 2139 agency centers,
  |v|_max = 2.815, 18786 lifecycle dissolutions

**Convergence-stop protocol (handover Part 6).** Active when `RUN_MODE == "verify-convergence"`.
Stops a phase when:

```
ma_delta < 0.001  AND  slope_recent < 0.0001  AND  lin >= 0.95  AND  ep >= min_epochs
```

with `min_epochs = 100` for Phase A and `50` for B/C/D; hard caps unchanged.

**Validation gate.** If avg lin shifts by more than ±0.01 from the headline (0.954) under
the convergence-stop protocol, the gate **fails**: the early-stop is too aggressive for
this regime, revert to current parameters and **do not** apply the protocol to C2-C8.


In [ ]:
# ════════════════════════════════════════════════════════════════
# § C1 — Local durable alignment without weight editing
# Layer: 1 (Existence)
# Foundational anchor: Foundational Claim 1 (Memory)
# Presupposes: nothing
# Falsifier: avg lin indistinguishable from base, OR retention decays under dormancy
# Artifacts: c1_canonical_lifecycle.json + .md, canonical_engine.pkl, canonical_metrics.pkl
# Convergence-stop: yes (when RUN_MODE == "verify-convergence")
# ════════════════════════════════════════════════════════════════
# This is the C1 validation gate for the entire convergence-stop protocol.
# If avg lin lands outside 0.954 ± 0.01, the protocol fails and must NOT
# be applied to C2-C8.
# ════════════════════════════════════════════════════════════════

import copy, pickle, math
import numpy as np
from scipy.stats import linregress

print("=" * 70)
print("  § C1 — CANONICAL LIFECYCLE TRAINING")
print("=" * 70)
print(f"  RUN_MODE = {RUN_MODE!r}")
print(f"  EARLY_STOP_STRONG_CONVERGE = {EARLY_STOP_STRONG_CONVERGE}")
print(f"  Target: avg lin = {HEADLINE_AVG_LIN} ± {HEADLINE_AVG_LIN_TOL}")
print()

# Per-claim seed (per handover Part 8).
C1_SEED = SEED + SEED_OFFSETS["C1"]
np.random.seed(C1_SEED); random.seed(C1_SEED)

# ----- Mode-driven hard caps + min_epochs -----
if RUN_MODE == "smoke":
    HARD_CAPS = {pn: 2 for pn in PHASE_NAMES}
    MIN_EPOCHS_PER_PHASE = {pn: 9999 for pn in PHASE_NAMES}  # disable early stop
    EVAL_INTERVAL = 1
elif RUN_MODE == "paper":
    HARD_CAPS = {"A_Onboarding": 300, "B_Initiative": 200,
                 "C_Reorg": 200, "D_Turnover": 200}
    MIN_EPOCHS_PER_PHASE = {pn: 9999 for pn in PHASE_NAMES}  # early-stop off
    EVAL_INTERVAL = 10
elif RUN_MODE == "verify-convergence":
    HARD_CAPS = {"A_Onboarding": 300, "B_Initiative": 200,
                 "C_Reorg": 200, "D_Turnover": 200}
    # Handover Part 6: min_epochs = 100 for A (conservative buffer), 50 for B/C/D.
    MIN_EPOCHS_PER_PHASE = {"A_Onboarding": 100, "B_Initiative": 50,
                            "C_Reorg": 50, "D_Turnover": 50}
    EVAL_INTERVAL = 5
else:
    raise RuntimeError(f"Unknown RUN_MODE: {RUN_MODE}")

print(f"  HARD_CAPS = {HARD_CAPS}")
print(f"  MIN_EPOCHS_PER_PHASE = {MIN_EPOCHS_PER_PHASE}")
print(f"  EVAL_INTERVAL = {EVAL_INTERVAL}")
print()

TRIM_D_HISTORY_TO = 100
C1_RESULTS_PATH = os.path.join(OUT_DIR, "c1_canonical_lifecycle.json")
C1_RESULTS_MD = os.path.join(OUT_DIR, "c1_canonical_lifecycle.md")
C1_ENGINE_PKL = os.path.join(OUT_DIR, "canonical_engine.pkl")
C1_METRICS_PKL = os.path.join(OUT_DIR, "canonical_metrics.pkl")
# Backward-compatible legacy aliases (the existing repo's downstream cells look
# at both names; new cells use the cN-prefixed names per handover Part 8).
C1_LEGACY_PATH = os.path.join(OUT_DIR, "canonical_training_results.json")


# ----- Reset engine for fresh canonical training -----
def _reset_engine_in_place(eng):
    eng.value_centers = []; eng.agency_centers = []
    eng.current_epoch = 0; eng.current_context_id = 0
    if hasattr(eng, "reset_verifications"):
        try:
            eng.reset_verifications()
        except Exception:
            pass
    eng._D_running_sum = 0.0
    eng._D_running_count = 0.0
    eng.set_context(0)
    if faiss_acc is not None:
        faiss_acc.rebuild_index()


_reset_engine_in_place(ibf)


# ----- Convergence-stop checker -----
def check_strong_convergence(acc_history, min_epochs):
    """Strong-convergence gate (handover Part 6).

    Returns (converged: bool, reason: str, slope: float, ma_delta: float).
    """
    if not acc_history or acc_history[-1][0] < min_epochs:
        return False, "below min_epochs", 0.0, 0.0
    if len(acc_history) < 10:
        return False, "insufficient history", 0.0, 0.0
    recent = acc_history[-10:]
    epochs = np.array([r[0] for r in recent], dtype=np.float64)
    accs = np.array([r[1] for r in recent], dtype=np.float64)
    first_half_ma = float(np.mean(accs[:5]))
    second_half_ma = float(np.mean(accs[5:]))
    ma_delta = abs(second_half_ma - first_half_ma)
    slope, *_ = linregress(epochs, accs)
    abs_slope = abs(slope)
    last_lin = float(accs[-1])
    # Strong convergence: ma_delta < 0.001 AND |slope| < 0.0001 AND lin >= 0.95
    if ma_delta < 0.001 and abs_slope < 0.0001 and last_lin >= 0.95:
        return True, (
            f"STRONG (ma_delta={ma_delta:.5f}, slope={slope:.7f}/ep, "
            f"lin={last_lin:.4f})"
        ), slope, ma_delta
    return False, (
        f"ma_delta={ma_delta:.5f}, slope={slope:.7f}/ep, lin={last_lin:.4f}"
    ), slope, ma_delta


# ----- Eval helpers -----
def _base_accuracy(pk):
    d = precomputed[f"{pk}_test"]
    rb, y = d["R_base_probs"], d["labels"]
    base_log = sum(
        1 for i in range(len(y))
        if np.argmax([np.log(rb[i, j] + 1e-8) for j in range(N_CHOICES)]) == y[i]
    ) / len(y)
    base_lin = float((rb.argmax(1) == y).mean())
    return float(base_log), float(base_lin)


def eval_phase(eng, pk, ctx):
    """Returns (acc_log, acc_lin)."""
    d = precomputed[f"{pk}_test"]
    zch, rb, y = d["z_choices"], d["R_base_probs"], d["labels"]
    eng.set_context(ctx)
    cor_log = cor_lin = 0
    for i in range(len(y)):
        dR = np.array([eng.delta_R(zch[i, j]) for j in range(N_CHOICES)])
        sc_log = np.array([np.log(rb[i, j] + 1e-8) + dR[j] for j in range(N_CHOICES)])
        sc_lin = np.array([rb[i, j] + dR[j] for j in range(N_CHOICES)])
        if int(np.argmax(sc_log)) == int(y[i]):
            cor_log += 1
        if int(np.argmax(sc_lin)) == int(y[i]):
            cor_lin += 1
    n = len(y)
    return float(cor_log / n), float(cor_lin / n)


def _center_stats(eng):
    nv = len(eng.value_centers); na = len(eng.agency_centers)
    nc = sum(1 for c in eng.value_centers if c.is_crystallized())
    vs = [abs(float(getattr(c, "v", 0.0))) for c in eng.value_centers]
    return {
        "n_value_centers": int(nv), "n_agency_centers": int(na),
        "n_crystallized": int(nc),
        "v_abs_mean": float(np.mean(vs)) if vs else 0.0,
        "v_abs_max": float(np.max(vs)) if vs else 0.0,
    }


# ----- Main training loop -----
all_diags, all_evals, phase_list = [], [], []
convergence_log = {}
base_acc_log, base_acc_lin = {}, {}

for pn in PHASE_NAMES:
    base_acc_log[pn], base_acc_lin[pn] = _base_accuracy(pn)

t0g = time.time()
total_epochs_used = 0
total_epochs_capped = 0  # if early-stop disabled, this counts hard-cap saturations

for pidx, pname in enumerate(PHASE_NAMES):
    hard_cap = HARD_CAPS[pname]
    min_ep = MIN_EPOCHS_PER_PHASE[pname]

    print(f"\n  {'═' * 60}")
    print(f"  PHASE {pidx}: {pname}  (cap={hard_cap}, min_ep={min_ep})")
    print(f"  {'═' * 60}")

    ibf.set_context(pidx)
    if pidx > 0:
        ibf.reset_verifications()
    ibf._D_running_sum = 0.0
    ibf._D_running_count = float("inf")  # disable D-centering

    d = precomputed[f"{pname}_train"]
    zq = d["z_question"]; zch = d["z_choices"]
    rb = d["R_base_probs"]; y = d["labels"]
    n = len(y)
    phase_list.append((pname, pidx))

    al, ai = eval_phase(ibf, pname, pidx)
    print(f"  base/current: log={al:.4f}, lin={ai:.4f}  |  base-only lin={base_acc_lin[pname]:.4f}")

    acc_history = []
    converged = False
    convergence_epoch = None
    final_slope = 0.0

    ep = 0
    while ep < hard_cap:
        perm = np.random.permutation(n)
        et0 = time.time()
        for idx in perm:
            zi_q = zq[idx]; zi_ch = zch[idx]
            ri = rb[idx]; yi = int(y[idx])
            for j in range(N_CHOICES):
                global _ADAPTER_R_FIELD_VALUE
                _ADAPTER_R_FIELD_VALUE = float(ri[j])
                R_truth = 0.0 if j == yi else 1.0
                ibf.compute_D_and_update(
                    board_before=zi_q, board_after_own_move=zi_ch[j],
                    board_after_opponent=None, move_chosen=j,
                    R_imposed_override=R_truth,
                )
        diag = ibf.end_epoch()
        all_diags.append(diag)
        if faiss_acc is not None:
            faiss_acc.rebuild_index()
        for c in ibf.value_centers + ibf.agency_centers:
            if len(c.D_history) > TRIM_D_HISTORY_TO:
                c.D_history = c.D_history[-TRIM_D_HISTORY_TO:]
        ep += 1

        if ep % EVAL_INTERVAL == 0 or ep == 1:
            ibf.set_context(pidx)
            cur_log, cur_lin = eval_phase(ibf, pname, pidx)
            acc_history.append((ep, cur_lin))
            elapsed = time.time() - t0g
            st = _center_stats(ibf)
            conv_marker = ""
            if EARLY_STOP_STRONG_CONVERGE:
                conv, reason, slope, ma_d = check_strong_convergence(acc_history, min_ep)
                if conv:
                    conv_marker = " ◆CONVERGED-STRONG"
            print(
                f"    Ep{ep:>3d}: "
                f"val={st['n_value_centers']}c/{st['n_crystallized']}x "
                f"agn={st['n_agency_centers']} "
                f"|D|={float(diag.get('D_abs_mean', 0.0)):.4f} "
                f"|v|={st['v_abs_mean']:.3f}/{st['v_abs_max']:.3f} "
                f"diss={int(diag.get('n_dissolved', 0))} "
                f"log={cur_log:.4f} lin={cur_lin:.4f} "
                f"{time.time()-et0:.1f}s [{elapsed:.0f}s]"
                f"{conv_marker}"
            )
            if EARLY_STOP_STRONG_CONVERGE:
                conv, reason, slope, ma_d = check_strong_convergence(acc_history, min_ep)
                if conv and not converged:
                    converged = True; convergence_epoch = ep; final_slope = slope
                    print(f"    ✓ Strong convergence at epoch {ep}: {reason}")
                    break

    total_epochs_used += ep
    if not converged:
        total_epochs_capped += 1
        print(f"\n    Phase {pname} stopped at cap ({hard_cap}); final lin = "
              f"{acc_history[-1][1] if acc_history else 0.0:.4f}")

    convergence_log[pname] = {
        "converged_strong": bool(converged),
        "convergence_epoch": convergence_epoch,
        "hard_cap": int(hard_cap),
        "epochs_run": int(ep),
        "final_slope": float(final_slope),
        "final_lin_acc": float(acc_history[-1][1]) if acc_history else 0.0,
    }

    # End-of-phase evaluation across all phases seen so far.
    ev = {}
    for pn2, ci2 in phase_list:
        ev[pn2] = eval_phase(ibf, pn2, ci2)
    all_evals.append({"after": pname, "accs": ev})
    print(f"\n  After {pname}:")
    for pn2, (a_log, a_lin) in ev.items():
        bl, bi = base_acc_log[pn2], base_acc_lin[pn2]
        marker = "▲" if (a_log - bl > 0.005 or a_lin - bi > 0.005) else "·"
        print(f"    {pn2}: log={a_log:.4f}(Δ{a_log-bl:+.4f})  "
              f"lin={a_lin:.4f}(Δ{a_lin-bi:+.4f}) {marker}")


# ----- Final metrics -----
print(f"\n  {'═' * 60}")
print("  FINAL METRICS (CANONICAL)")
print(f"  {'═' * 60}")
final_train = {pn: eval_phase(ibf, pn, i) for i, pn in enumerate(PHASE_NAMES)}
final_rows = []
for pn in PHASE_NAMES:
    al, ai = final_train[pn]
    bl, bi = base_acc_log[pn], base_acc_lin[pn]
    final_rows.append({
        "phase": pn,
        "base_log": float(bl), "ibf_log": float(al), "delta_log": float(al - bl),
        "base_lin": float(bi), "ibf_lin": float(ai), "delta_lin": float(ai - bi),
    })
    print(f"  {pn:<20s} base_lin={bi:.4f}  ibf_lin={ai:.4f}  Δ={ai-bi:+.4f}")

avg_bl = float(np.mean(list(base_acc_log.values())))
avg_bi = float(np.mean(list(base_acc_lin.values())))
avg_al = float(np.mean([v[0] for v in final_train.values()]))
avg_ai = float(np.mean([v[1] for v in final_train.values()]))

st_final = _center_stats(ibf)
total_dissolutions = int(sum(int(d_.get("n_dissolved", 0)) for d_ in all_diags))


# ════════════════════════════════════════════════════════════════
# VALIDATION GATE — handover Part 6
# ════════════════════════════════════════════════════════════════
# When RUN_MODE == "verify-convergence", check whether avg lin lands
# within ±0.01 of the headline (0.954). If not, the convergence-stop
# protocol fails and must NOT be applied to C2-C8.
# ════════════════════════════════════════════════════════════════
measured_avg_lin = avg_ai
deviation = measured_avg_lin - HEADLINE_AVG_LIN
within_tolerance = abs(deviation) <= HEADLINE_AVG_LIN_TOL

print(f"\n  {'═' * 60}")
print("  C1 VALIDATION GATE  (handover Part 6)")
print(f"  {'═' * 60}")
print(f"    EXPECTED   avg lin = {HEADLINE_AVG_LIN:.4f}  ± {HEADLINE_AVG_LIN_TOL}")
print(f"    GOT        avg lin = {measured_avg_lin:.4f}  (Δ = {deviation:+.4f})")
print(f"    WITHIN_TOLERANCE   = {within_tolerance}")

CONVERGENCE_GATE_PASSED = None
if RUN_MODE == "verify-convergence":
    CONVERGENCE_GATE_PASSED = bool(within_tolerance)
    if within_tolerance:
        print(f"    ◆ GATE PASSED — early-stop protocol approved for C2-C8")
    else:
        print(f"    ✗ GATE FAILED — early-stop too aggressive; do NOT apply to C2-C8")
        print(f"      ALERT: revert to RUN_MODE='paper' for C2-C8 and document.")
elif RUN_MODE == "paper":
    print(f"    (paper mode — gate is informational only; no early-stop was active)")
elif RUN_MODE == "smoke":
    print(f"    (smoke mode — gate not applicable)")


# ----- Save artifacts -----
c1_results = {
    "claim": "C1",
    "claim_short": "local_durable_alignment_without_weight_editing",
    "layer": 1,
    "foundational_anchor": "Foundational Claim 1 (Memory)",
    "engine_version": ENGINE_VERSION,
    "run_mode": RUN_MODE,
    "seed": C1_SEED,
    "model": model_name,
    "geometry": {
        "sigma_operating": float(SIGMA_PROP),
        "sigma_agency": float(SIGMA_AGENCY),
        "merge_threshold": float(MERGE_PROP),
    },
    "base_lin": {k: float(v) for k, v in base_acc_lin.items()},
    "base_log": {k: float(v) for k, v in base_acc_log.items()},
    "ibf_lin": {k: float(v[1]) for k, v in final_train.items()},
    "ibf_log": {k: float(v[0]) for k, v in final_train.items()},
    "final_rows": final_rows,
    "average": {
        "base_lin": avg_bi, "ibf_lin": avg_ai, "delta_lin": avg_ai - avg_bi,
        "base_log": avg_bl, "ibf_log": avg_al, "delta_log": avg_al - avg_bl,
    },
    "convergence_log": convergence_log,
    "n_value_centers": int(st_final["n_value_centers"]),
    "n_agency_centers": int(st_final["n_agency_centers"]),
    "n_crystallized": int(st_final["n_crystallized"]),
    "v_abs_mean": float(st_final["v_abs_mean"]),
    "v_abs_max": float(st_final["v_abs_max"]),
    "total_dissolutions": total_dissolutions,
    "runtime_minutes": float((time.time() - t0g) / 60.0),
    "validation_gate": {
        "expected_avg_lin": HEADLINE_AVG_LIN,
        "tolerance": HEADLINE_AVG_LIN_TOL,
        "measured_avg_lin": float(measured_avg_lin),
        "deviation": float(deviation),
        "within_tolerance": bool(within_tolerance),
        "gate_passed": CONVERGENCE_GATE_PASSED,
        "applies_to_C2_C8_globally": (
            CONVERGENCE_GATE_PASSED if CONVERGENCE_GATE_PASSED is not None
            else "n/a (gate only active in verify-convergence mode)"
        ),
    },
    "early_stop_strong_converge": EARLY_STOP_STRONG_CONVERGE,
    "total_epochs_used": int(total_epochs_used),
    "total_phases_capped": int(total_epochs_capped),
}
_write_json(C1_RESULTS_PATH, c1_results)
_write_json(C1_LEGACY_PATH, c1_results)  # backward-compat alias
print(f"\n  Saved: {C1_RESULTS_PATH}")
print(f"  Saved legacy alias: {C1_LEGACY_PATH}")

# Engine pickle for downstream cells.
canonical_engine = copy.deepcopy(ibf)
canonical_engine_payload = {
    "value_centers": ibf.value_centers,
    "agency_centers": ibf.agency_centers,
    "current_epoch": ibf.current_epoch,
    "params": ibf.p,
    "precomputed": precomputed,
    "pca": pca, "scaler": scaler,
    "subject_to_id": subject_to_id, "answer_to_id": answer_to_id,
    "phase_data": phase_data,
    "SIGMA_PROP": float(SIGMA_PROP), "SIGMA_AGENCY": float(SIGMA_AGENCY),
    "MERGE_PROP": float(MERGE_PROP),
    "Z_DIM": Z_DIM, "Z_DIM_AUG": Z_DIM_AUG,
    "SUBJECT_DIM": SUBJECT_DIM, "ANSWER_DIM": ANSWER_DIM,
    "SUBJECT_SCALE": SUBJECT_SCALE, "ANSWER_SCALE": ANSWER_SCALE,
    "convergence_log": convergence_log,
    "all_evals": all_evals,
    "c1_results": c1_results,
    "engine_version": ENGINE_VERSION,
}
with open(C1_ENGINE_PKL, "wb") as f:
    pickle.dump(canonical_engine_payload, f)
print(f"  Saved engine pkl: {C1_ENGINE_PKL}")
canonical_metrics = {
    "all_evals": all_evals, "convergence_log": convergence_log,
    "base_acc_log": base_acc_log, "base_acc_lin": base_acc_lin,
    "final_train": final_train, "final_rows": final_rows,
}
with open(C1_METRICS_PKL, "wb") as f:
    pickle.dump(canonical_metrics, f)
print(f"  Saved metrics pkl: {C1_METRICS_PKL}")

# Markdown report.
with open(C1_RESULTS_MD, "w", encoding="utf-8") as f:
    f.write("# § C1 — Canonical lifecycle training\n\n")
    f.write(f"**Engine:** {ENGINE_VERSION}\n")
    f.write(f"**Run mode:** {RUN_MODE}\n\n")
    f.write("## Headline\n\n")
    f.write(f"- avg lin = {avg_ai:.4f} (base {avg_bi:.4f}, Δ {avg_ai-avg_bi:+.4f})\n")
    f.write(f"- expected = {HEADLINE_AVG_LIN} ± {HEADLINE_AVG_LIN_TOL}; within = {within_tolerance}\n")
    f.write(f"- value centers: {st_final['n_value_centers']} ({st_final['n_crystallized']} crystallised)\n")
    f.write(f"- agency centers: {st_final['n_agency_centers']}\n")
    f.write(f"- |v|_max = {st_final['v_abs_max']:.4f}\n")
    f.write(f"- dissolutions: {total_dissolutions}\n")
    f.write("\n## Per-phase\n\n")
    f.write("| phase | base_lin | ibf_lin | Δ |\n|---|---:|---:|---:|\n")
    for r in final_rows:
        f.write(f"| {r['phase']} | {r['base_lin']:.4f} | {r['ibf_lin']:.4f} | {r['delta_lin']:+.4f} |\n")
    f.write("\n## Validation gate\n\n")
    f.write(f"- EXPECTED avg lin = `{HEADLINE_AVG_LIN}` ± `{HEADLINE_AVG_LIN_TOL}`\n")
    f.write(f"- GOT avg lin = `{measured_avg_lin:.4f}` (Δ `{deviation:+.4f}`)\n")
    f.write(f"- WITHIN_TOLERANCE = `{within_tolerance}`\n")
    if CONVERGENCE_GATE_PASSED is not None:
        f.write(f"- CONVERGENCE_GATE_PASSED = `{CONVERGENCE_GATE_PASSED}`\n")
print(f"  Saved markdown: {C1_RESULTS_MD}")

print(f"\n  ✓ C1 complete.  runtime = {(time.time()-t0g)/60:.1f} min")
